# Lab 6: pn Junction — Poisson Equation and Diode Characteristics

Electrostatics of the abrupt pn junction from the Poisson equation, compared with the depletion approximation, and its current–voltage characteristic (Neamen Chs. 7–8). All four modules of the simulator work together here.

Update history
- 2026.10.12 : General-purpose 1D drift–diffusion simulator: Scharfetter–Gummel discretization, Gummel iteration, SRH recombination, bias continuation, comparison with analytic models, Plotly visualization.
- 2026.10.13 : Launcher with case presets and device schematics; one reverse-to-forward bias sweep; 2kT/q-corrected depletion approximation and equivalent box width; temperature control.
- 2026.10.20 : Rebuilt on the modules of `semi1d` (`statistics`, `drift_diffusion`, `continuity`, `poisson`); the Gummel coupling is written out in this notebook; textbook assumptions listed.

ref
1. D. A. Neamen, *Semiconductor Physics and Devices: Basic Principles*, 4th ed., McGraw-Hill (2012), Chs. 3–8.
2. S. M. Sze and K. K. Ng, *Physics of Semiconductor Devices*, 3rd ed., John Wiley & Sons (2007), Ch. 2.
3. H. K. Gummel, *A self-consistent iterative scheme for one-dimensional steady state transistor calculations*, IEEE Transactions on Electron Devices **11**, 455–465 (1964).
4. D. L. Scharfetter and H. K. Gummel, *Large-signal analysis of a silicon Read diode oscillator*, IEEE Transactions on Electron Devices **16**, 64–77 (1969). [DOI](https://doi.org/10.1109/T-ED.1969.16566).
5. S. Selberherr, *Analysis and Simulation of Semiconductor Devices*, Springer (1984). [DOI](https://doi.org/10.1007/978-3-7091-8752-4).
6. Plotly, [Multiple legends](https://plotly.com/python/legend/) and [Animations in Python](https://plotly.com/python/animations/).

---

## Usage Guide

### Run a calculation

1. Execute all code cells. The first code cell imports the four modules of `semi1d` from the folder above this notebook; the last cell, **(4) Simulation**, displays the launcher.
2. Select a **Case**; its preset parameters are loaded, a one-line description shows what the case compares, and a schematic to the right of the parameters shows the junction (p blue, n red) and, for the bias sweep, the terminal **V** on the p-side contact and the grounded n-side contact. Edit any parameter, then click **Run Interact**.

The output below the launcher lists the device and grid, the Gummel convergence of each bias, a table comparing simulated quantities with analytic models, and one interactive Plotly figure. The last result is stored in `LAST['res']`. Positions are in µm, concentrations in cm⁻³, energies in eV, potentials in V, fields in V/cm, current densities in A/cm² and lifetimes in µs.

### Modules used in this lab

| Module | Neamen | Equation | Lab 4 | Lab 5 | **Lab 6** |
| --- | --- | --- | --- | --- | --- |
| `statistics` | Chs. 3–4 | Boltzmann densities, $n_i(T)$, quasi-Fermi levels | ✓ | ✓ | **✓** |
| `drift_diffusion` | Ch. 5 | $J_n$, $J_p$ = drift + diffusion | ✓ | ✓ | **✓** |
| `continuity` | Ch. 6 | $(1/q)\,dJ_p/dx=G-R$, SRH | – | ✓ | **✓** |
| `poisson` | Ch. 7 | $d^2\psi/dx^2=-(e/\varepsilon_s)(p-n+N_d-N_a)$ (7.11) | – | – | **✓** |

Labs 4 and 5 assume quasi-neutrality, as Neamen does in Chs. 5 and 6; the pn junction has a space-charge region, so this lab solves the Poisson equation.

### Cases

| Case | What it shows |
| --- | --- |
| **6-A** pn junction at equilibrium | Charge, field and potential vs the depletion approximation |
| **6-B** Symmetric vs asymmetric doping | Where the depletion approximation fails |
| **6-C** Bias sweep | $J$–$V$, ideality factor, $W$ and $\lvert\mathcal{E}\rvert_{max}$ from reverse to forward bias |

### Case parameters

| Symbol | Control | Meaning and unit |
| --- | --- | --- |
| $N_a$, $N_d$ | **Na, p side**, **Nd, n side** | Acceptor and donor concentrations, cm⁻³ |
| — | **Set 1–3: Na, Nd** | Doping sets of 6-B, e.g. `1e17, 1e15` |
| $L_p$, $L_n$ | **p-side length**, **n-side length** | Lengths of the p and n regions, µm |
| $V_{R,max}$, $\Delta V_R$ | **Maximum reverse bias**, **VR step** | Reverse part of the sweep, $V=-V_{R,max},\ldots,-\Delta V_R,0$, V |
| $V_{max}$, $\Delta V$ | **Maximum forward bias**, **Forward bias step** | Forward part of the sweep, $V=\Delta V,\ldots,V_{max}$, V |
| — | **Initial guess at each bias** | Previous bias solution (continuation) or equilibrium |
| $\tau$ | **τn = τp** | SRH lifetimes, µs |
| — | **Recombination (SRH)** | On: SRH with lifetime $\tau$; Off: $R=0$ |
| $\theta$ | **Depletion edge \|ρ\|/(qN)** | Simulated depletion edges are where $\lvert\rho\rvert>\theta qN$ |
| $T$ | **Temperature** | Lattice temperature, 200–500 K; changes $kT/e$ and $n_i(T)$ only, K |
| $N_x$ | **Grid points (0 = auto)** | Number of grid points; 0 resolves the Debye length automatically |

### Read the output

- **Title:** the case ID, doping, material, temperature and bias.
- **Position:** $x=0$ is the metallurgical junction; the p-side contact is at $x=-L_p$ and the n-side contact at $x=+L_n$.
- **Charge, field and potential:** $\rho/q=p-n+N_d-N_a$, $\mathcal{E}=-d\psi/dx$ and $\psi$; the charge and field panels open around the depletion region (use Plotly zoom or double-click to rescale); the depletion approximation is drawn in gray.
- **Energy band diagram and concentrations:** $E_c$, $E_v$, $E_{Fi}$, $E_{Fn}$ (blue dashed), $E_{Fp}$ (red dashed); electrons blue, holes red, doping dotted.
- **Bias sweep (6-C):** the upper panels show $J$–$V$ on linear and log axes, the local ideality factor, $W$, $\lvert\mathcal{E}\rvert_{max}$ and $W^2$; the orange marker shows the bias selected with the slider, and the lower panels show the internal profiles at that bias. **Play** steps through the biases.
- In the comparison table, *Diff.* is $(\text{sim}-\text{analytic})/\lvert\text{analytic}\rvert$; "—" means no analytic counterpart or a model that does not apply. Equation numbers refer to [1].

## Basic Theory

### All modules together

$$\text{Poisson (7.11):}\quad\frac{d^2\psi}{dx^2}=-\frac{e}{\varepsilon_s}\left(p-n+N_d-N_a\right),\qquad \mathcal{E}=-\frac{d\psi}{dx}$$

$$\text{Currents (5.35):}\quad J_n=e\mu_nn\mathcal{E}+eD_n\frac{dn}{dx},\qquad J_p=e\mu_pp\mathcal{E}-eD_p\frac{dp}{dx};\qquad \text{continuity (Sec. 6.2.1):}\quad \frac{1}{e}\frac{dJ_n}{dx}=R-G,\quad \frac{1}{e}\frac{dJ_p}{dx}=G-R$$

with SRH recombination $R=(np-n_i^2)/[\tau_{p0}(n+n_i)+\tau_{n0}(p+n_i)]$ (6.99) and Boltzmann densities $n=n_ie^{(E_{Fn}-E_{Fi})/kT}$, $p=n_ie^{(E_{Fi}-E_{Fp})/kT}$ (6.85). Both contacts are ohmic: $\psi$, $n$ and $p$ keep their equilibrium values, so excess carriers vanish there. The bias $V$ is applied to the p-side (left) contact and the n-side contact is grounded, so $V>0$ is forward bias and $V<0$ reverse bias ($V_R=-V$). The built-in potential is set by the equilibrium potentials of the two contacts, so it agrees with (7.10) by construction; $W$, $\lvert\mathcal{E}\rvert_{max}$ and $\rho(x)$ are the real tests of the depletion approximation.

### Textbook assumptions

| Assumption | Where Neamen uses it | This simulation |
| --- | --- | --- |
| **Boltzmann approximation** (3.80) and **complete ionization** (Sec. 4.4.2), stated again for the pn junction in Sec. 7.2 | $V_{bi}$ (7.10), densities at the depletion edges (Sec. 8.1.3) | Kept (`statistics`); fails for degenerate doping or at low temperature |
| **Step junction** with uniform doping on each side (Sec. 7.1) | Charge densities (7.12) | Kept |
| **Abrupt depletion approximation**: the space charge ends abruptly at $-x_p$ and $x_n$, neutral outside (Sec. 7.2.2) | $x_n$, $x_p$, $W$, $\lvert\mathcal{E}\rvert_{max}$ (7.28)–(7.36) | **Relaxed**: the Poisson equation is solved with mobile carriers (Debye tails, $2kT/e$ correction) |
| Ideal diode, assumption 1 (Sec. 8.1.2): abrupt depletion layer, neutral outside | (8.26), (8.27) | Relaxed (as above) |
| Ideal diode, assumption 2: Maxwell–Boltzmann statistics | Boundary conditions (Sec. 8.1.3) | Kept |
| Ideal diode, assumption 3: low injection and complete ionization | Minority diffusion in the neutral regions | Low injection **not assumed** (row `injection_V`); complete ionization kept |
| Ideal diode, assumptions 4a, 4b: total current constant; $J_n$, $J_p$ continuous | Current at the depletion edges | Satisfied by the continuity equations |
| Ideal diode, assumption 4c: $J_n$, $J_p$ constant through the depletion region (no G–R there) | (8.26), (8.27) | **Relaxed**: SRH acts in the depletion region; compared with $J_{rec}$ (8.55) and the total (8.56) |
| Recombination current with maximum rate across $W$, $\tau_0=(\tau_{n0}+\tau_{p0})/2$ | $J_{rec}$ (8.55) | The simulation computes $R(x)$ directly; (8.55) is only an estimate |

**Simulation model.** One dimension; steady state; one lattice temperature; constant $E_g$, $\varepsilon_r$ and mobilities; $n_i(T)=n_i(300)(T/300)^{3/2}e^{-E_g/2k(1/T-1/300)}$; SRH with a midgap trap; no Auger, radiative recombination, avalanche or tunneling (the reverse current never breaks down); ohmic contacts with no resistance; uniform grid.

### Analytic models used for comparison

| Structure / condition | Quantity | Analytic model [1] |
| --- | --- | --- |
| pn junction | Built-in potential | $V_{bi}=(kT/e)\ln(N_aN_d/n_i^2)$ (7.10) |
| pn junction, $V<V_{bi}$ | Depletion approximation | $x_n$ (7.28), $x_p$ (7.29), $W$ (7.31), (7.34), $\lvert\mathcal{E}\rvert_{max}$ (7.36) with $V_{bi}-V$ |
| pn junction, $V<V_{bi}$ | Majority-carrier tails | $W$ and $\lvert\mathcal{E}\rvert_{max}$ with $V_{bi}-V-2kT/e$ [2, Sec. 2.2]; compared with the equivalent box width $\int\lvert\rho\rvert\,dx/qN$ |
| pn junction | Debye length | $L_D=(\varepsilon_skT/e^2N)^{1/2}$ |
| pn junction, $V\ne0$ | Ideal diode | $J=J_s(e^{eV/kT}-1)$ (8.27); $J_s=\frac{eD_nn_{p0}}{L_n}\coth\frac{W_p}{L_n}+\frac{eD_pp_{n0}}{L_p}\coth\frac{W_n}{L_p}$, long-diode limit (8.26) |
| pn junction, $V\ne0$ | Minority carriers at the depletion edges | $\delta n_p(-x_p)=n_{p0}(e^{eV/kT}-1)$, $\delta p_n(x_n)=p_{n0}(e^{eV/kT}-1)$ (Sec. 8.1.3) |
| pn junction, $V\ne0$ | Depletion-region recombination / generation | $J_{rec}=\frac{eWn_i}{2\tau_0}e^{eV/2kT}$ ($V>0$) (8.55); $J_{gen}=\frac{eWn_i}{2\tau_0}$ ($V<0$); total (8.56) |

Here $W_p$ and $W_n$ are the neutral-region widths between the depletion edges and the contacts; $\coth\to1$ for a long diode ($W\gg L$) and $\coth(W/L)\to L/W$ for a short diode ($W\ll L$). With SRH off, $L\to\infty$: $J_s=eD_nn_{p0}/W_p+eD_pp_{n0}/W_n$ and $J_{rec}=J_{gen}=0$.

<details>
<summary><strong>Advanced Theory</strong></summary>

<h3>Grid and units</h3>

<p>The junction is sampled on a uniform grid $x_i=i\,\Delta x$. With <strong>Grid points = 0</strong> the spacing resolves the smallest Debye length, $\Delta x\le L_D(N_{max})/8$, with $401\le N\le20001$. The printed ratio $\Delta x/L_D$ should stay well below 1.</p>

<h3>Scharfetter–Gummel discretization (modules <code>drift_diffusion</code>, <code>continuity</code>)</h3>

<p>With $\Delta_i=(\psi_{i+1}-\psi_i)/V_t$ and $B(z)=z/(e^z-1)$, $J_{n,i+1/2}=(qD_n/\Delta x)[B(\Delta_i)n_{i+1}-B(-\Delta_i)n_i]$ and $J_{p,i+1/2}=(qD_p/\Delta x)[B(\Delta_i)p_i-B(-\Delta_i)p_{i+1}]$ [4, 5]. This exponential fitting is exact for a constant field between nodes and stays stable when $\psi$ changes by many $V_t$ over one interval. With $\psi$ fixed, each continuity equation is a linear tridiagonal system; SRH is linearized with its denominator fixed.</p>

<h3>Poisson equation (module <code>poisson</code>)</h3>

<p>With the quasi-Fermi levels frozen, $n=n^{(k)}e^{(\psi-\psi^{(k)})/V_t}$ and $p=p^{(k)}e^{-(\psi-\psi^{(k)})/V_t}$, the Poisson equation is nonlinear in $\psi$ and is solved by Newton iteration with a tridiagonal Jacobian; each update is damped as $\delta\psi\to\mathrm{sign}(\delta\psi)V_t\ln(1+\lvert\delta\psi\rvert/V_t)$. The equilibrium solution starts from local charge neutrality and solves the Poisson–Boltzmann equation.</p>

<h3>Gummel iteration and bias continuation (this notebook)</h3>

<p>(1) Poisson, (2) hole continuity, (3) electron continuity, repeated until $\max\lvert\Delta\psi\rvert<10^{-9}$ V and $\max\lvert\Delta n\rvert/(n+n_i)$, $\max\lvert\Delta p\rvert/(p+n_i)<10^{-7}$ (at most 3000 iterations, <code>SIM_CFG</code>) [3]. In a sweep with continuation, the biases are solved outward from $V=0$; each bias starts from the previous solution with its potential shifted by $\Delta V(1-x/L)$, using sub-steps of at most 0.05 V in forward and 0.25 V in reverse bias. Gummel iteration converges quickly at low currents and needs more iterations as the current grows.</p>

<h3>Derived quantities</h3>

<ul>
<li>Simulated depletion edges are the outermost points with $\lvert\rho\rvert>\theta qN$; the equivalent box width $\int\lvert\rho\rvert\,dx/qN$ does not depend on $\theta$.</li>
<li>Minority-carrier tail lengths are exponential fits over $0.3L<d<2.5L$ from the depletion edges, extrapolated to the edges.</li>
<li>The local ideality factor is $n=(1/V_t)\,dV/d(\ln J)$ for $V>0$.</li>
<li>The total current is a small difference of large drift and diffusion currents; its numerical noise floor is about $10^{-10}$ A/cm² for a silicon junction (row <code>Jt_max</code> at equilibrium).</li>
</ul>

</details>

<details>
<summary><strong>Code Implementation</strong></summary>

<h3>Modules and notebook code</h3>

<table>
<thead><tr><th>Where</th><th>Names</th><th>Role</th></tr></thead>
<tbody>
<tr><td><code>semi1d.statistics</code></td><td><code>thermal_voltage</code>, <code>intrinsic_density</code>, <code>boltzmann_densities</code>, <code>band_diagram</code></td><td>$kT/e$, $n_i(T)$, Boltzmann densities, band edges and quasi-Fermi levels</td></tr>
<tr><td><code>semi1d.drift_diffusion</code></td><td><code>currents</code>, <code>flux_operator</code></td><td>Scharfetter–Gummel currents and their divergence</td></tr>
<tr><td><code>semi1d.continuity</code></td><td><code>solve_continuity</code>, <code>srh_rate</code>, <code>diffusion_length</code>, <code>finite_decay</code></td><td>Continuity equations with SRH, minority profiles</td></tr>
<tr><td><code>semi1d.poisson</code></td><td><code>equilibrium</code>, <code>solve_poisson</code>, <code>built_in_potential</code>, <code>depletion_width</code>, <code>max_field</code>, <code>depletion_profiles</code>, <code>debye_length</code></td><td>Poisson equation and the depletion approximation of Ch. 7</td></tr>
<tr><td>This notebook</td><td><code>MATERIALS</code>, <code>Device</code>, <code>build_junction</code></td><td>Material parameters, grid and doping</td></tr>
<tr><td>This notebook</td><td><code>GummelSolver</code> (<code>solve</code>, <code>sweep</code>)</td><td>Coupling of the four modules, bias continuation</td></tr>
<tr><td>This notebook</td><td><code>ideal_diode_Js</code>, <code>J_rec</code>, <code>J_gen</code>, <code>depletion_compare</code>, <code>analyze_junction</code>, <code>analyze_sweep</code></td><td>Ideal diode of Ch. 8, comparison rows</td></tr>
<tr><td>This notebook</td><td><code>run_junction</code>, <code>run_doping_sets</code>, <code>CASES</code>, <code>run_case</code>, <code>list_cases</code>, <code>plot_result</code>, <code>launch</code></td><td>Runners, case presets, figures and launcher</td></tr>
</tbody>
</table>

<p>A result is a <code>SimulationResult</code>: <code>res.table</code> holds every comparison row (<code>res.value('W')</code>, <code>res.value('W', 'ref')</code>); for one bias <code>res.data['state']</code> holds the profiles (<code>x</code>, <code>xc</code>, <code>psi</code>, <code>n</code>, <code>p</code>, <code>E</code>, <code>rho</code>, <code>R</code>, <code>Ec</code>, <code>Ev</code>, <code>Fn</code>, <code>Fp</code>, currents); for a sweep <code>res.data</code> holds <code>V</code>, <code>J</code>, <code>iters</code>, <code>ideality</code>, <code>W_sim</code>, <code>W_box</code>, <code>W_dep</code>, <code>W_2kT</code>, <code>Emax_sim</code>, <code>Emax_dep</code>, <code>Emax_2kT</code>, <code>J_diode</code>, <code>J_rg</code>, <code>J_model</code>. After editing a module in <code>semi1d/</code>, restart the kernel.</p>

<pre><code>res = run_case('6-C', verbose=False)
V, W = res.data['V'], res.data['W_box']                # V, um (equivalent box width)
m = V <= 0                                              # reverse-bias part
slope, intercept = np.polyfit(-V[m], W[m]**2, 1)        # W^2 = slope * V_R + intercept
plot_result(res)

for T in [250.0, 300.0, 350.0]:
    r = run_case('6-A', T=T, verbose=False)
    print(T, r.value('Vbi'), r.value('W', 'ref'))
</code></pre>

</details>

---
# (1) Calculation

Modules of `semi1d` used by this lab, then the parts written in this notebook: units, materials and device; the lab model (how the modules are coupled) and its analysis; runners and case presets.

In [ ]:
###############################################################################
# Imports: all four modules of semi1d (folder one level above this notebook)
###############################################################################

import os
import re
import sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
from dataclasses import dataclass, field
from types import SimpleNamespace
from scipy import constants as sc_const

from semi1d import statistics          # Ch. 3-4: n_i(T), Boltzmann densities, quasi-Fermi levels
from semi1d import drift_diffusion     # Ch. 5:   drift and diffusion currents, Einstein relation
from semi1d import continuity          # Ch. 6:   continuity equations, SRH recombination
from semi1d import poisson             # Ch. 7:   Poisson equation, depletion approximation

In [ ]:
###############################################################################
# Units, Materials and Device
###############################################################################
# Internal units: length [cm], potential [V], energy [eV], time [s], density [cm^-3]; positions shown in [um].

Q = sc_const.e                          # Elementary charge [C]
EPS0 = sc_const.epsilon_0 * 1e-2        # Vacuum permittivity [F/cm]
UM2CM, US2S = 1e-4, 1e-6


@dataclass(frozen=True)
class Material:
    name: str
    eps_r: float        # Relative permittivity
    Eg: float           # Band gap [eV]
    ni300: float        # Intrinsic carrier concentration at 300 K [cm^-3]
    mun: float          # Electron mobility [cm^2/V s]
    mup: float          # Hole mobility [cm^2/V s]


# 300 K values: Neamen (2012), Table 5.1 (mobility) and Appendix B.4
MATERIALS = {
    'Si':   Material('Si',   11.7, 1.12, 1.5e10, 1350.0,  480.0),
    'Ge':   Material('Ge',   16.0, 0.66, 2.4e13, 3900.0, 1900.0),
    'GaAs': Material('GaAs', 13.1, 1.42, 1.8e6,  8500.0,  400.0),
}


@dataclass
class GridConfig:
    points_per_debye: float = 8.0       # Auto grid: dx <= L_D(min) / points_per_debye
    npts_min: int = 401                 # Auto grid: lower limit of grid points
    npts_max: int = 20001               # Auto grid: upper limit of grid points
    npts_uniform: int = 1001            # Auto grid for a uniform bar (no built-in space charge)


GRID_CFG = GridConfig()


class Device:
    """
    1D device: grid x [um] (measured from the left contact), doping N_a(x), N_d(x) [cm^-3], material and
    transport parameters (n_i(T) from semi1d.statistics, D from the Einstein relation in semi1d.drift_diffusion).
    x0_um is the plotting origin (junction or device center); srh=False removes SRH recombination (R = 0).
    bc_left / bc_right: 'ohmic' contact (equilibrium concentrations) or 'free' surface (no carrier flux).
    """

    def __init__(self, x_um, Na, Nd, material='Si', T=300.0, ni300=None, mun=None, mup=None,
                 taun=1e-7, taup=1e-7, srh=True, x0_um=None, bc_left='ohmic', bc_right='ohmic'):
        mat = MATERIALS[material] if isinstance(material, str) else material
        self.material = mat
        self.x_um = np.asarray(x_um, dtype=float)
        self.x = self.x_um * UM2CM                                  # [cm]
        self.N = len(self.x)
        self.dx = self.x[1] - self.x[0]
        self.Na = np.broadcast_to(np.asarray(Na, dtype=float), self.x.shape).copy()
        self.Nd = np.broadcast_to(np.asarray(Nd, dtype=float), self.x.shape).copy()
        self.C = self.Nd - self.Na                                  # Net doping
        self.T = float(T)
        self.Vt = statistics.thermal_voltage(self.T)
        self.Eg = mat.Eg
        self.ni = statistics.intrinsic_density(self.T, mat.Eg, mat.ni300 if ni300 is None else float(ni300))
        self.mun = mat.mun if mun is None else float(mun)
        self.mup = mat.mup if mup is None else float(mup)
        self.Dn, self.Dp = drift_diffusion.einstein(self.mun, self.Vt), drift_diffusion.einstein(self.mup, self.Vt)
        self.taun, self.taup = float(taun), float(taup)
        self.srh = bool(srh)
        self.tau_eff_n = self.taun if self.srh else np.inf      # Used by the analytic models
        self.tau_eff_p = self.taup if self.srh else np.inf
        self.eps = mat.eps_r * EPS0
        self.x0_um = 0.5 * self.x_um[-1] if x0_um is None else float(x0_um)
        if bc_left not in ('ohmic', 'free') or bc_right not in ('ohmic', 'free'):
            raise ValueError("Ends must be 'ohmic' (contact) or 'free' (no surface recombination).")
        self.bc_left, self.bc_right = bc_left, bc_right


def debye_length_um(N, eps, Vt):
    """Extrinsic Debye length (eps kT / e^2 N)^{1/2} [um] (used to choose the grid)."""
    return np.sqrt(eps * Vt / (Q * N)) / UM2CM


def grid_points(uniform, L_um, Nmax, eps, Vt, npts=None):
    """Grid points: npts, or automatic (1001 for a uniform bar; otherwise dx <= L_D(Nmax)/8, 401-20001)."""
    if npts is None:
        if uniform:
            return GRID_CFG.npts_uniform
        LD = debye_length_um(Nmax, eps, Vt)
        npts = int(np.clip(int(np.ceil(GRID_CFG.points_per_debye * L_um / LD)) + 1, GRID_CFG.npts_min,
                           GRID_CFG.npts_max))
    if int(npts) < 11:
        raise ValueError("Use at least 11 grid points.")
    return int(npts)


def check_positive(**values):
    bad = [k for k, v in values.items() if not np.isfinite(v) or v <= 0]
    if bad:
        raise ValueError(f"Lengths, temperature, lifetimes and doping concentrations must be positive: {bad}")


def print_device(dev, structure_name, say, origin='device center'):
    """Device and grid block of the log."""
    LD_min = debye_length_um(float(np.max(np.maximum(dev.Na, dev.Nd))), dev.eps, dev.Vt)
    say('(0) Device and grid')
    say(f'structure \t = {structure_name}, {dev.material.name}, T = {dev.T:g} K')
    say(f'device length \t = {dev.x_um[-1]:.3f} µm, number of grid = {dev.N}, dx = {dev.dx / UM2CM * 1e3:.3f} nm')
    say(f'x range \t = {-dev.x0_um:.3f} to {dev.x_um[-1] - dev.x0_um:.3f} µm (x = 0 at the {origin})')
    say(f'Eg, ni, kT/e \t = {dev.Eg:.3g} eV, {dev.ni:.4e} cm^-3, {dev.Vt:.5f} V')
    say(f'μn, μp \t = {dev.mun:g}, {dev.mup:g} cm^2/V s  (Dn, Dp = {dev.Dn:.4g}, {dev.Dp:.4g} cm^2/s)')
    return LD_min


###############################################################################
# Comparison Table and Results
###############################################################################

ITER_ROWS = ('iters',)        # Printed first in every table


def _is_num(v):
    return isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, bool) and np.isfinite(v)


def plain_sub(s):
    """Subscripts in printed text, where they cannot be typeset: 'E_g', 'μ_n', 'N_{a}' -> 'Eg', 'μn', 'Na'."""
    return re.sub(r'(?<=[^\W\d_])_(\{[^}]*\}|[A-Za-z0-9]+)', lambda m: m.group(1).strip('{}'), s) if isinstance(s, str) else s


def row(key, name, sim, ref=None, model=''):
    """One line of the comparison table. diff = (sim - ref)/|ref| when both are numbers. The key is the name used
    by res.value(key); subscripts in the quantity and the model are written without underscores (plain_sub)."""
    sim = float(sim) if _is_num(sim) else sim
    ref = float(ref) if _is_num(ref) else ref
    diff = (sim - ref) / abs(ref) if _is_num(sim) and _is_num(ref) and ref != 0 else None
    return dict(key=key, name=plain_sub(name), sim=sim, ref=ref, diff=diff, model=plain_sub(model))


def _fmt(v, width=12):
    if isinstance(v, str):
        s = v
    elif not _is_num(v):                          # None or NaN
        s = '—'
    elif isinstance(v, (int, np.integer)):
        s = f'{v:d}'
    else:
        s = f'{v:.4g}'
    return f'{s:>{width}s}'


def select_rows(rows, keys=None):
    """Rows to print: iteration rows first, then the requested keys in the given order (None = all rows)."""
    if keys is None:
        return rows
    by_key = {r['key']: r for r in rows}
    out = [by_key[k] for k in ITER_ROWS if k in by_key]
    out += [by_key[k] for k in keys if k in by_key and k not in ITER_ROWS]
    missing = [k for k in keys if k not in by_key]
    if missing:
        print(f'(not applicable for these parameters: {", ".join(missing)})')
    return out


def print_table(rows):
    """Print the comparison table: Key | Quantity | Simulation | Analytic | Diff. | Model"""
    print(f"{'Key':<14s} {'Quantity':<46s} {'Simulation':>12s} {'Analytic':>12s} {'Diff.':>8s}  Model")
    print('-' * 118)
    for r in rows:
        diff = f"{100 * r['diff']:+.1f}%" if r['diff'] is not None else '—'
        print(f"{r['key']:<14s} {r['name']:<46s} {_fmt(r['sim'])} {_fmt(r['ref'])} {diff:>8s}  {r['model']}")


@dataclass
class SimulationResult:
    """Result of a case: res.table (comparison rows), res.data (arrays), res.value(key)."""
    params: dict
    device: object
    states: list
    table: list
    data: dict = field(default_factory=dict)
    case: str = None
    panels: list = None

    def row(self, key):
        for r in self.table:
            if r['key'] == key:
                return r
        raise KeyError(f"No table row '{key}'. Available keys: {[r['key'] for r in self.table]}")

    def value(self, key, which='sim'):
        """Table value by key; which = 'sim', 'ref' or 'diff'."""
        return self.row(key)[which]


###############################################################################
# Case Runner
###############################################################################
# CASES = {case_id: dict(title, summary, params, ranges, fixed, runner, rows, cols, panels, schematic)}

def to_float(text, name='value'):
    try:
        return float(str(text).strip())
    except ValueError:
        raise ValueError(f"'{name}' must be a number such as 1e16; got {text!r}.") from None


def run_case(case_id, verbose=True, **overrides):
    """Run a case, e.g. run_case(list(CASES)[0], T=350.0). Parameters not given keep their presets."""
    case = CASES[case_id]
    bad = set(overrides) - set(case['params'])
    if bad:
        raise KeyError(f"Case {case_id} has no parameter {sorted(bad)}; available: {list(case['params'])}")
    kw = {**case.get('fixed', {}), **case_kwargs({**case['params'], **overrides})}
    if verbose:
        print(f"Case {case_id}: {case['title']}\n")
    if 'cols' in case:
        kw['cols'] = case['cols']
    res = case['runner'](verbose=verbose, rows=case.get('rows'), **kw)
    res.case, res.panels = case_id, case.get('panels')
    return res


def list_cases():
    """Print the case IDs, titles and parameter presets."""
    for cid, case in CASES.items():
        ps = ', '.join(f'{k}={v:g}' if isinstance(v, float) else f'{k}={v!r}' for k, v in case['params'].items())
        print(f"{cid}  {case['title']}\n      run_case('{cid}', {ps})")

In [ ]:
###############################################################################
# Lab 6 Device: abrupt pn junction (p for 0 <= x < L_p, n for L_p <= x <= L_p + L_n)
###############################################################################

STRUCTURES = {3: 'Abrupt pn junction'}


def build_junction(Na=1e16, Nd=1e15, Lp_um=2.0, Ln_um=5.0, material='Si', T=300.0, ni300=None, mun=None, mup=None,
                   taun_us=0.1, taup_us=0.1, srh=True, npts=None):
    """Abrupt pn junction on a uniform grid; x = 0 of the plots at the junction (x0_um = L_p)."""
    L_um = Lp_um + Ln_um
    check_positive(L_um=L_um, T=T, taun_us=taun_us, taup_us=taup_us, Lp_um=Lp_um, Ln_um=Ln_um, Na=Na, Nd=Nd)
    mat = MATERIALS[material]
    npts = grid_points(False, L_um, float(max(Na, Nd)), mat.eps_r * EPS0, statistics.thermal_voltage(T), npts)
    x = np.linspace(0.0, L_um, npts)
    return Device(x, np.where(x < Lp_um, Na, 0.0), np.where(x >= Lp_um, Nd, 0.0), material=mat, T=T, ni300=ni300,
                  mun=mun, mup=mup, taun=taun_us * US2S, taup=taup_us * US2S, srh=srh, x0_um=Lp_um)


###############################################################################
# Coupling of the Modules: Gummel Iteration
###############################################################################
# (1) poisson.solve_poisson with frozen quasi-Fermi levels      -> psi
# (2) continuity.solve_continuity for holes with psi fixed       -> p   (drift-diffusion flux + G - R)
# (3) continuity.solve_continuity for electrons with psi fixed   -> n
# (4) repeat until psi, n and p stop changing

@dataclass
class SolverConfig:
    gummel_maxit: int = 3000            # Maximum number of Gummel iterations
    tol_psi: float = 1e-9               # Convergence: max|d psi| [V]
    tol_carrier: float = 1e-7           # Convergence: max|dn|/(n + n_i), max|dp|/(p + n_i)
    poisson_maxit: int = 200            # Maximum number of Newton iterations in Poisson
    poisson_tol: float = 1e-11          # Newton tolerance [V]
    max_forward_step: float = 0.05      # Bias continuation sub-step for V > 0 [V]
    max_reverse_step: float = 0.25      # Bias continuation sub-step for V <= 0 [V]


SIM_CFG = SolverConfig()


class GummelSolver:
    """Self-consistent solution of the Poisson, current and continuity equations (all four modules).
    The equilibrium solution (psi_eq, n_eq, p_eq) is the Poisson-Boltzmann solution of semi1d.poisson."""

    def __init__(self, dev):
        self.d = dev
        self.psi_eq, self.n_eq, self.p_eq = poisson.equilibrium(dev, SIM_CFG.poisson_maxit, SIM_CFG.poisson_tol)

    def solve(self, V=0.0, G=0.0, inj=0.0, guess=None, maxit=None):
        """Steady state at the bias V of the left (p-side) contact; the right contact is grounded."""
        d = self.d
        Vt = d.Vt
        G = np.broadcast_to(np.asarray(G, dtype=float), (d.N,)).copy()
        if guess is None:
            psi = self.psi_eq + V * (1.0 - d.x / d.x[-1])
            n, p = self.n_eq.copy(), self.p_eq.copy()
        else:
            psi, n, p = (np.array(a, dtype=float) for a in guess)
        if d.bc_left == 'ohmic':
            psi[0], n[0], p[0] = self.psi_eq[0] + V, self.n_eq[0] + inj, self.p_eq[0] + inj
        if d.bc_right == 'ohmic':
            psi[-1], n[-1], p[-1] = self.psi_eq[-1], self.n_eq[-1], self.p_eq[-1]
        lo, hi = poisson.span(d)
        hist, converged = [], False
        for it in range(1, (maxit or SIM_CFG.gummel_maxit) + 1):
            psi0, n0, p0 = psi.copy(), n.copy(), p.copy()
            psi = poisson.solve_poisson(d, psi, n, p, psi0, SIM_CFG.poisson_maxit, SIM_CFG.poisson_tol)   # (1)
            u = np.clip((psi - psi0) / Vt, -80, 80)[lo:hi]
            n[lo:hi] *= np.exp(u)
            p[lo:hi] *= np.exp(-u)
            p = continuity.solve_continuity(d, psi, n, p, G, 'p')                                          # (2)
            n = continuity.solve_continuity(d, psi, n, p, G, 'n')                                          # (3)
            dv = float(np.max(np.abs(psi - psi0)))
            dn = float(np.max(np.abs(n - n0) / (n0 + d.ni)))
            dp = float(np.max(np.abs(p - p0) / (p0 + d.ni)))
            hist.append((it, dv, dn, dp))
            if dv < SIM_CFG.tol_psi and dn < SIM_CFG.tol_carrier and dp < SIM_CFG.tol_carrier:        # (4)
                converged = True
                break
        return self.pack(V, psi, n, p, G, inj, hist, converged)

    def sweep(self, Vlist, from_equilibrium=False, callback=None, **kw):
        """Several biases, solved outward from V = 0. With continuation, each bias starts from the previous one
        (sub-steps of at most max_forward_step / max_reverse_step; potential guess shifted by dV (1 - x/L))."""
        d = self.d
        ramp = 1.0 - d.x / d.x[-1]
        res = {}
        for sgn in (1, -1):
            targets = sorted({float(v) for v in Vlist if (v > 0 if sgn > 0 else v <= 0)}, key=abs)
            step = SIM_CFG.max_forward_step if sgn > 0 else SIM_CFG.max_reverse_step
            guess, vcur = None, 0.0
            for vt in targets:
                if from_equilibrium:
                    st = self.solve(vt, **kw)
                else:
                    nstep = max(1, int(np.ceil(abs(vt - vcur) / step - 1e-9)))
                    vprev = vcur
                    for k in range(1, nstep + 1):
                        vnext = vt if k == nstep else vcur + (vt - vcur) * k / nstep
                        if guess is not None:
                            guess = (guess[0] + (vnext - vprev) * ramp, guess[1], guess[2])
                        st = self.solve(vnext, guess=guess, **kw)
                        guess, vprev = (st.psi, st.n, st.p), vnext
                    vcur = vt
                res[vt] = st
                if callback is not None:
                    callback(vt, st)
        return [res[float(v)] for v in Vlist]

    def pack(self, V, psi, n, p, G, inj, hist, converged):
        d = self.d
        J = drift_diffusion.currents(d, psi, n, p)
        bands = statistics.band_diagram(psi, n, p, d.ni, d.Vt, d.Eg)
        xh = 0.5 * (d.x_um[1:] + d.x_um[:-1])
        return SimpleNamespace(
            V=V, x=d.x_um, xh=xh, xc=d.x_um - d.x0_um, xhc=xh - d.x0_um, psi=psi, n=n, p=p,
            dn=n - self.n_eq, dp=p - self.p_eq, E=poisson.electric_field(d, psi), rho=poisson.space_charge(d, n, p),
            R=continuity.srh_rate(d, n, p), G=G, **bands, **vars(J), J=float(np.mean(J.Jt)), hist=hist,
            iters=len(hist), converged=converged, inj=inj, dev=d)


###############################################################################
# Ideal Diode (Neamen Ch. 8)
###############################################################################

def ideal_diode_Js(Na, Nd, ni, Dn, Dp, taun, taup, Wp=np.inf, Wn=np.inf):
    """Ideal-diode saturation current [A/cm^2] with neutral widths W_p, W_n and ohmic contacts:
    J_s = e D_n n_p0/L_n coth(W_p/L_n) + e D_p p_n0/L_p coth(W_n/L_p). Long diode (W >> L): (8.26); short diode
    (W << L): L replaced by W. The diode current is J = J_s [exp(eV/kT) - 1] (8.27)."""
    def term(D, n0, L, W):
        if not np.isfinite(L):                    # No recombination: short-diode limit D n0 / W
            return D * n0 / W if np.isfinite(W) else 0.0
        return D * n0 / L * (1.0 if not np.isfinite(W) else 1.0 / np.tanh(W / L))
    return Q * (term(Dn, ni ** 2 / Na, np.sqrt(Dn * taun), Wp) + term(Dp, ni ** 2 / Nd, np.sqrt(Dp * taup), Wn))


def J_rec(W, ni, tau0, V, Vt):
    """Forward-bias recombination current J_rec = (e W n_i / 2 tau_0) exp(eV/2kT) [A/cm^2], Neamen (8.55)."""
    return Q * W * ni / (2 * tau0) * np.exp(V / (2 * Vt))


def J_gen(W, ni, tau0):
    """Reverse-bias generation current J_gen = e n_i W / (2 tau_0) [A/cm^2]."""
    return Q * ni * W / (2 * tau0)


###############################################################################
# Lab 6 Analysis
###############################################################################

def _len(v):
    return f'{v:.3g}' if np.isfinite(v) else '∞'


def dep_edges(st, xj, thr):
    """Depletion edges from |rho| > thr q N [um]. output: x_p, x_n (distances from x_j)"""
    x, rho, d = st.x, st.rho, st.dev
    mp = (x < xj) & (rho < -thr * d.Na)
    mn = (x >= xj) & (rho > thr * d.Nd)
    xp = xj - x[mp].min() if mp.any() else 0.0
    xn = x[mn].max() - xj if mn.any() else 0.0
    return float(xp), float(xn)


def tail_fit(dist, y, L):
    """Exponential fit of a neutral-region tail (0.3 L < dist < 2.5 L), extrapolated to dist = 0.
    output: decay length [um], value at dist = 0"""
    sel = (dist > 0.3 * L) & (dist < 2.5 * L) & (y > 0)
    if sel.sum() < 3:
        return np.nan, np.nan
    c = np.polyfit(dist[sel], np.log(y[sel]), 1)
    return -1.0 / c[0], float(np.exp(c[1]))


def depletion_compare(st, Na, Nd, xj, thr, V=0.0):
    """Simulated and depletion-approximation x_n, x_p, W [um] and |E|max [V/cm] at bias V (V_R = -V)."""
    d = st.dev
    Vbi = poisson.built_in_potential(Na, Nd, d.ni, d.Vt)
    Vtot = Vbi - V
    valid = Vtot > 4 * d.Vt                      # Depletion approximation is meaningless near V = V_bi
    W = poisson.depletion_width(Na, Nd, Vtot, d.eps) / UM2CM if valid else np.nan
    xp_s, xn_s = dep_edges(st, xj, thr)
    # Equivalent box width: depleted charge on each side divided by its doping
    rp = np.where((st.x < xj) & (st.rho < 0), -st.rho, 0.0)
    rn = np.where((st.x >= xj) & (st.rho > 0), st.rho, 0.0)
    W_box = (rp.sum() / Na + rn.sum() / Nd) * d.dx / UM2CM
    return dict(Vbi=Vbi, valid=valid, W=W, xp_a=W * Nd / (Na + Nd), xn_a=W * Na / (Na + Nd),
                xp_s=xp_s, xn_s=xn_s, W_s=xp_s + xn_s, W_box=W_box,
                W_2kT=poisson.depletion_width(Na, Nd, Vtot - 2 * d.Vt, d.eps) / UM2CM if valid else np.nan,
                Em_a=poisson.max_field(Na, Nd, Vtot, d.eps) if valid else np.nan,
                Em_2kT=poisson.max_field(Na, Nd, Vtot - 2 * d.Vt, d.eps) if valid else np.nan,
                Em_s=float(np.max(np.abs(st.E))))


###############################################################################
# Lab 6 Analysis (continued)
###############################################################################

def analyze_junction(st, s, p):
    """
    Compare one operating point of the pn junction with the depletion approximation (Ch. 7) and the ideal diode
    (Ch. 8).

    output:
        list    rows    Comparison-table rows
        dict    ov      Analytic overlays for plotting {panel: [(x, y, label, dash), ...]}
    """
    d = st.dev
    x, L = st.x, st.x[-1]
    V, G, inj = st.V, float(np.max(st.G)), st.inj
    equilibrium = (V == 0 and G == 0 and inj == 0)
    rows, ov = [], {k: [] for k in ('E', 'rho', 'psi', 'excess')}

    Na, Nd, xj = p['Na'], p['Nd'], p['Lp_um']
    c = depletion_compare(st, Na, Nd, xj, p['thr'], V)
    LDp = poisson.debye_length(Na, d.eps, d.Vt) / UM2CM
    LDn = poisson.debye_length(Nd, d.eps, d.Vt) / UM2CM
    Ln = continuity.diffusion_length(d.Dn, d.tau_eff_n) / UM2CM
    Lp = continuity.diffusion_length(d.Dp, d.tau_eff_p) / UM2CM
    rows += [row('Vbi', 'Built-in potential ψ_eq(L) − ψ_eq(0) [V]', s.psi_eq[-1] - s.psi_eq[0], c['Vbi'],
                 'kT/e ln(NaNd/ni²) (7.10)'),
             row('ni', 'Intrinsic concentration n_i [cm⁻³]', d.ni, None, f'T = {d.T:g} K'),
             row('xn', 'n-side depletion width x_n [µm]', c['xn_s'], c['xn_a'], 'depletion approx. (7.28)'),
             row('xp', 'p-side depletion width x_p [µm]', c['xp_s'], c['xp_a'], 'depletion approx. (7.29)'),
             row('W', 'Depletion width W [µm]', c['W_s'], c['W'], 'depletion approx. (7.31), (7.34)'),
             row('Emax', 'Maximum field |E|max [V/cm]', c['Em_s'], c['Em_a'], 'depletion approx. (7.36)'),
             row('W_box', 'Equivalent box width ∫|ρ|dx/(qN) [µm]', c['W_box'], c['W_2kT'],
                 'depletion approx., V_bi − V − 2kT/q [2]'),
             row('Emax_2kT', 'Maximum field |E|max [V/cm]', c['Em_s'], c['Em_2kT'],
                 'depletion approx., V_bi − V − 2kT/q [2]'),
             row('LD_p', 'Debye length, p side [µm]', LDp, None, 'L_D = (εkT/e²N_a)^1/2'),
             row('LD_n', 'Debye length, n side [µm]', LDn, None, 'L_D = (εkT/e²N_d)^1/2'),
             row('xp_LD', 'x_p/L_D(p), x_n/L_D(n)', f"{c['xp_a'] / LDp:.3g}, {c['xn_a'] / LDn:.3g}", None,
                 'approx. degrades near 1')]
    if c['valid']:
        psi_p = st.psi[0]
        rho_a, E_a, psi_a = poisson.depletion_profiles((x - xj) * UM2CM, Na, Nd, c['xp_a'] * UM2CM, c['xn_a'] * UM2CM,
                                                    psi_p, d.eps)
        ov['rho'].append((x, rho_a, 'depletion approx.', 'dash'))
        ov['E'].append((x, E_a, 'depletion approx.', 'dash'))
        ov['psi'].append((x, psi_a, 'depletion approx.', 'dash'))
    # Minority carriers in the neutral regions and the ideal-diode current
    xp_e = c['xp_a'] if c['valid'] else c['xp_s']
    xn_e = c['xn_a'] if c['valid'] else c['xn_s']
    Wp, Wn = xj - xp_e, (L - xj) - xn_e                     # Neutral widths [um]
    np0, pn0 = d.ni ** 2 / Na, d.ni ** 2 / Nd
    loj = np.expm1(V / d.Vt)
    Js = ideal_diode_Js(Na, Nd, d.ni, d.Dn, d.Dp, d.tau_eff_n, d.tau_eff_p, Wp * UM2CM, Wn * UM2CM)
    rows += [row('Ln_Lp', 'Diffusion lengths L_n, L_p [µm]', f'{_len(Ln)}, {_len(Lp)}', None,
                 'L = (Dτ)^1/2' if d.srh else 'SRH off: no recombination'),
             row('W_neutral', 'Neutral widths p side, n side [µm]', f'{Wp:.3g}, {Wn:.3g}', None,
                 'long diode if ≫ L'),
             row('Js', 'Ideal-diode saturation current J_s [A/cm²]', None, Js, 'finite widths; long diode (8.26)')]
    if V != 0 and Wp > 0 and Wn > 0:
        mp, mn = x <= xj - xp_e, x >= xj + xn_e
        sp, sn = (xj - xp_e) - x[mp], x[mn] - (xj + xn_e)     # Distances from the depletion edges [um]
        ov['excess'] += [(x[mp], np.abs(np0 * loj * continuity.finite_decay(sp, Wp, Ln)), 'ideal diode δn_p', 'dash'),
                         (x[mn], np.abs(pn0 * loj * continuity.finite_decay(sn, Wn, Lp)), 'ideal diode δp_n', 'dot')]
        JD = Js * loj
        tau0 = 0.5 * (d.tau_eff_n + d.tau_eff_p)
        W_cm = c['W'] * UM2CM
        Jrg = J_rec(W_cm, d.ni, tau0, V, d.Vt) if V > 0 else -J_gen(W_cm, d.ni, tau0)
        rows += [row('J', 'Current density J [A/cm²]', st.J, JD, 'ideal diode (8.27)'),
                 row('J_rg', 'Depletion-region recomb./gen. current [A/cm²]', None, Jrg,
                     'J_rec (8.55) or −J_gen' if d.srh else 'SRH off: 0'),
                 row('J_model', 'Current density J [A/cm²]', st.J, JD + Jrg, 'ideal diode + J_rg (8.56)')]
        if V > 0:
            Lfn, An = tail_fit(sp, st.n[mp] - np0, Ln)
            Lfp, Ap = tail_fit(sn, st.p[mn] - pn0, Lp)
            src_n = 'tail fit' if np.isfinite(An) else 'direct'
            src_p = 'tail fit' if np.isfinite(Ap) else 'direct'
            if not np.isfinite(An):
                An = float(np.interp(xj - xp_e, x, st.n)) - np0
            if not np.isfinite(Ap):
                Ap = float(np.interp(xj + xn_e, x, st.p)) - pn0
            rows += [row('L_tail_n', 'p-side electron tail length [µm]', Lfn, Ln, 'L_n = (D_n τ_n)^1/2'),
                     row('L_tail_p', 'n-side hole tail length [µm]', Lfp, Lp, 'L_p = (D_p τ_p)^1/2'),
                     row('dn_edge', 'δn_p at −x_p [cm⁻³]', An, np0 * loj, f'Sec. 8.1.3; sim: {src_n}'),
                     row('dp_edge', 'δp_n at x_n [cm⁻³]', Ap, pn0 * loj, f'Sec. 8.1.3; sim: {src_p}'),
                     row('dEFn_dep', 'E_Fn(−x_p) − E_Fn(x_n) [eV]',
                         float(np.interp(xj - xp_e, x, st.Fn) - np.interp(xj + xn_e, x, st.Fn)), 0.0,
                         'flat quasi-Fermi levels'),
                     row('injection_V', 'Injection level δn_p(−x_p)/N_a', An / Na, '≪ 1', 'low injection, Sec. 6.3.2')]

    # ---------------- Space charge and current components ----------------
    Nloc = np.maximum(np.maximum(d.Na, d.Nd), d.ni)
    viol = np.abs(st.rho) > 0.1 * Nloc
    rows += [row('Jn_drift_max', 'max |J_n drift| [A/cm²]', np.max(np.abs(st.Jn_drift))),
             row('Jn_diff_max', 'max |J_n diffusion| [A/cm²]', np.max(np.abs(st.Jn_diff))),
             row('Jp_drift_max', 'max |J_p drift| [A/cm²]', np.max(np.abs(st.Jp_drift))),
             row('Jp_diff_max', 'max |J_p diffusion| [A/cm²]', np.max(np.abs(st.Jp_diff))),
             row('Jt_max', 'max |J_n + J_p| [A/cm²]', np.max(np.abs(st.Jt)),
                 '0' if equilibrium else None, 'equilibrium (5.43)' if equilibrium else ''),
             row('qn_violation', 'Quasi-neutrality violation max |ρ|/(qN)', np.max(np.abs(st.rho) / Nloc),
                 '≪ 1', 'quasi-neutrality, Sec. 5.3.1'),
             row('rho_width', 'Width of the |ρ| > 0.1qN region [µm]',
                 (x[viol].max() - x[viol].min()) if viol.any() else 0.0, None, 'space-charge region')]
    return rows, ov

In [ ]:
###############################################################################
# Lab 6 Runners
###############################################################################

def sweep_data(Vs, states):
    """Per-bias arrays of a bias sweep: V, J, iterations, convergence and the local ideality factor."""
    J = np.array([st.J for st in states])
    ideality = np.full(len(Vs), np.nan)
    fw = (Vs > 0) & (J > 0)
    if fw.sum() >= 2:
        ideality[fw] = np.gradient(Vs[fw], np.log(J[fw])) / states[0].dev.Vt
    return dict(V=Vs, J=J, iters=np.array([st.iters for st in states]),
                converged=np.array([st.converged for st in states]), ideality=ideality, states=states)


def analyze_sweep(Vs, states, s, p):
    """Per-bias W, |E|max and diode-model currents of a bias sweep; rows of the table at V = 0."""
    d = s.d
    per_bias = [analyze_junction(st, s, p) for st in states]
    get = lambda rows, key: next((r for r in rows if r['key'] == key), None)
    data = dict(overlays=[ov for _, ov in per_bias])
    keys = dict(W_sim=('W', 'sim'), W_dep=('W', 'ref'), W_box=('W_box', 'sim'), W_2kT=('W_box', 'ref'),
                Emax_sim=('Emax', 'sim'), Emax_dep=('Emax', 'ref'), Emax_2kT=('Emax_2kT', 'ref'), Js=('Js', 'ref'))
    for name, (key, which) in keys.items():
        data[name] = np.array([get(rows, key)[which] for rows, _ in per_bias], dtype=float)
    data['J_diode'] = data['Js'] * np.expm1(Vs / d.Vt)
    data['J_rg'] = np.array([(get(rows, 'J_rg') or {}).get('ref', np.nan) for rows, _ in per_bias], dtype=float)
    data['J_rg'] = np.where(Vs == 0, 0.0, data['J_rg'])
    data['J_model'] = data['J_diode'] + np.nan_to_num(data['J_rg'])     # J_rg is undefined near V_bi
    rows0 = analyze_junction(s.solve(0.0), s, p)[0]
    return [get(rows0, k) for k in ('Vbi', 'Ln_Lp', 'W_neutral', 'Js', 'LD_p', 'LD_n')], data


def run_junction(Na=1e16, Nd=1e15, Lp_um=2.0, Ln_um=5.0, material='Si', T=300.0, ni300=None, mun=None, mup=None,
                 taun_us=0.1, taup_us=0.1, srh=True, analysis='point', V=0.0, V_list=None, from_equilibrium=False,
                 thr=0.5, npts=None, verbose=True, rows=None, cols=None):
    """
    Abrupt pn junction: one operating point (analysis='point', bias V) or a bias sweep (analysis='sweep', V_list).

    output:
        SimulationResult        res (res.data['state'] for one point; per-bias arrays for a sweep)
    """
    params = dict(structure=3, Na=Na, Nd=Nd, Lp_um=Lp_um, Ln_um=Ln_um, material=material, T=T, ni300=ni300,
                  mun=mun, mup=mup, taun_us=taun_us, taup_us=taup_us, srh=srh, analysis=analysis, V=V,
                  V_list=V_list, from_equilibrium=from_equilibrium, thr=thr, npts=npts, G=0.0, inj=0.0)
    if not 0 < thr < 1:
        raise ValueError("Use 0 < thr < 1.")
    say = print if verbose else (lambda *a, **k: None)
    dev = build_junction(Na, Nd, Lp_um, Ln_um, material, T, ni300, mun, mup, taun_us, taup_us, srh, npts)
    LD_min = print_device(dev, 'Abrupt pn junction', say, origin='junction')
    if srh:
        say(f'τn, τp \t = {taun_us:g}, {taup_us:g} us  (Ln, Lp = '
            f'{continuity.diffusion_length(dev.Dn, dev.taun) / UM2CM:.4g}, '
            f'{continuity.diffusion_length(dev.Dp, dev.taup) / UM2CM:.4g} µm)')
    else:
        say('recombination \t = SRH off (R = 0)')
    say(f'Debye length \t = {LD_min:.4g} µm (highest doping), dx / LD = {dev.dx / UM2CM / LD_min:.3g}')
    say('\n(1) Solving the Poisson, current and continuity equations (Gummel iteration) ...')
    s = GummelSolver(dev)
    report = lambda tag, st: say(f'\t{tag:<14s}: {"converged" if st.converged else "NOT converged"} '
                                 f'in {st.iters} iterations, J = {st.J:+.4e} A/cm^2')
    if analysis == 'point':
        st = s.solve(V)
        report(f'V = {V:+.3f} V', st)
        iters = row('iters', 'Gummel iterations', st.iters, None,
                    'converged' if st.converged else 'NOT converged (increase SIM_CFG.gummel_maxit)')
        table, ov = analyze_junction(st, s, params)
        res = SimulationResult(params, dev, [st], [iters] + table, dict(state=st, overlays=ov))
    else:
        Vs = np.unique(np.round(np.asarray(V_list, dtype=float), 6)) + 0.0       # + 0.0 removes -0.0
        if Vs.size == 0 or Vs.size > 101 or not np.all(np.isfinite(Vs)):
            raise ValueError("Use 1 to 101 finite bias points.")
        states = s.sweep(list(Vs), from_equilibrium=from_equilibrium,
                         callback=lambda v, st: report(f'V = {v:+.3f} V', st))
        data = sweep_data(Vs, states)
        extra_rows, extra = analyze_sweep(Vs, states, s, params)
        data.update(extra)
        table = [row('Vt', 'Thermal voltage kT/e [V]', dev.Vt), row('ni', 'Intrinsic concentration n_i [cm⁻³]', dev.ni)]
        res = SimulationResult(params, dev, states, table + extra_rows, data)
    say('Done!!')
    if verbose:
        print('\n(2) Comparison with analytic models')
        print_table(select_rows(res.table, rows))
        if analysis == 'sweep':
            print()
            print_sweep_table(res, cols)
    return res


# Sweep-table columns: key -> (header, values from res.data, format)
SWEEP_COLUMNS = {
    'V':        ('V [V]',       lambda d: d['V'],         '{:+.3f}'),
    'VR':       ('VR [V]',      lambda d: 0.0 - d['V'],   '{:.3f}'),
    'J':        ('J [A/cm²]',   lambda d: d['J'],         '{:.4e}'),
    'J_diode':  ('J ideal',     lambda d: d['J_diode'],   '{:.4e}'),
    'J_rg':     ('Jrg',         lambda d: d['J_rg'],      '{:.4e}'),
    'J_model':  ('J ideal+rg',  lambda d: d['J_model'],   '{:.4e}'),
    'ideality': ('n',           lambda d: d['ideality'],  '{:.3f}'),
    'W_sim':    ('W [µm]',      lambda d: d['W_sim'],     '{:.4g}'),
    'W_dep':    ('W dep.',      lambda d: d['W_dep'],     '{:.4g}'),
    'W_box':    ('W box',       lambda d: d['W_box'],     '{:.4g}'),
    'W_2kT':    ('W dep.-2kT',  lambda d: d['W_2kT'],     '{:.4g}'),
    'Emax_2kT': ('|E|max -2kT', lambda d: d['Emax_2kT'],  '{:.4g}'),
    'Emax_sim': ('|E|max',      lambda d: d['Emax_sim'],  '{:.4g}'),
    'Emax_dep': ('|E|max dep.', lambda d: d['Emax_dep'],  '{:.4g}'),
    'iters':    ('iters',       lambda d: d['iters'],     '{:.0f}'),
}


def print_sweep_table(res, cols=None):
    """Print per-bias results of a bias sweep; cols = list of SWEEP_COLUMNS keys (None = default set)."""
    d = res.data
    if cols is None:
        cols = ['V', 'J', 'J_diode', 'J_model', 'ideality', 'W_sim', 'W_dep', 'Emax_sim', 'Emax_dep', 'iters']
    cols = [c for c in cols if c == 'V' or c == 'VR' or c in d]
    order = np.argsort(-d['V'], kind='stable') if 'VR' in cols else np.arange(len(d['V']))   # V_R increasing
    print('Bias sweep (n: local ideality factor, dep.: depletion approximation)')
    print('  '.join(f'{SWEEP_COLUMNS[c][0]:>12s}' for c in cols))
    values = {c: np.asarray(SWEEP_COLUMNS[c][1](d), dtype=float) for c in cols}
    for k in order:
        line = [f'{SWEEP_COLUMNS[c][2].format(values[c][k]):>12s}' if np.isfinite(values[c][k]) else f"{'—':>12s}"
                for c in cols]
        print('  '.join(line) + ('' if d['converged'][k] else '  (NOT converged)'))


def run_doping_sets(sets=((1e16, 1e16), (1e16, 1e15), (1e17, 1e15)), Lp_um=1.0, Ln_um=5.0, thr=0.5,
                    npts=None, verbose=True, rows=None, **kw):
    """
    Compare abrupt pn junctions with several doping sets at equilibrium (depletion approximation vs simulation).

    output:
        SimulationResult        res, with res.data['sets'] = [{label, Na, Nd, state, E_dep, Emax_dep, result}]
    """
    say = print if verbose else (lambda *a, **k: None)
    say('(0)-(1) Equilibrium solution for each doping set (Gummel iteration) ...')
    table, data_sets = [], []
    for k, (Na, Nd) in enumerate(sets, 1):
        r = run_junction(Na=Na, Nd=Nd, Lp_um=Lp_um, Ln_um=Ln_um, thr=thr, npts=npts, verbose=False, **kw)
        st, dev = r.data['state'], r.device
        LD = debye_length_um(max(Na, Nd), dev.eps, dev.Vt)
        label = f'set {k}: {Na:.0e} / {Nd:.0e}'
        say(f'\t{label:<24s}: number of grid = {dev.N}, dx / LD = {dev.dx / UM2CM / LD:.3g}, '
            f'{"converged" if st.converged else "NOT converged"} in {st.iters} iterations')
        table += [row(f'W_set{k}', f'Set {k}: depletion width W [µm]', r.value('W'), r.value('W', 'ref'),
                      'depletion approx. (7.31)'),
                  row(f'Emax_set{k}', f'Set {k}: maximum field |E|max [V/cm]', r.value('Emax'),
                      r.value('Emax', 'ref'), 'depletion approx. (7.36)'),
                  row(f'xp_LD_set{k}', f'Set {k}: x_p / L_D(p)', r.value('xp', 'ref') / r.value('LD_p'), None,
                      'approx. degrades near 1')]
        E_ov = r.data['overlays']['E']
        data_sets.append(dict(label=label, Na=Na, Nd=Nd, state=st, result=r, Emax_dep=r.value('Emax', 'ref'),
                              E_dep=E_ov[0][1] if E_ov else np.full(dev.N, np.nan)))
    say('Done!!')
    params = dict(structure=3, analysis='doping_sets', sets=[tuple(s) for s in sets], Lp_um=Lp_um, Ln_um=Ln_um,
                  thr=thr, npts=npts, **kw)
    res = SimulationResult(params, data_sets[0]['state'].dev, [ds['state'] for ds in data_sets], table,
                           dict(sets=data_sets))
    if verbose:
        print('\n(2) Comparison with analytic models')
        print_table(select_rows(res.table, rows))
    return res


###############################################################################
# Lab 6 Cases
###############################################################################

def case_kwargs(p):
    """Translate case parameters into runner arguments (lifetimes, bias lists, doping sets, automatic grid)."""
    kw = dict(p)
    if 'tau_us' in kw:                                        # tau_n = tau_p
        kw['taun_us'] = kw['taup_us'] = kw.pop('tau_us')
    biases = []
    if 'VR_max' in kw:                                        # Reverse bias V_R = V_R,max ... 0 (V = -V_R)
        step, vr = kw.pop('VR_step'), kw.pop('VR_max')
        if step <= 0 or vr < 0:
            raise ValueError('Use a positive V_R step and V_R,max >= 0.')
        n = int(np.floor(vr / step + 1e-9))
        biases += list(-step * np.arange(n, -1, -1))
    if 'V_max' in kw:                                         # Forward bias V = V_step ... V_max
        step, vmax = kw.pop('V_step'), kw.pop('V_max')
        if step <= 0 or vmax <= 0:
            raise ValueError('Use a positive forward step and maximum forward bias.')
        biases += list(step * np.arange(1, int(np.floor(vmax / step + 1e-9)) + 1))
    if biases:
        kw['V_list'] = biases
    sets = [kw.pop(k) for k in ('set1', 'set2', 'set3') if k in kw]
    if sets:
        kw['sets'] = []
        for k, text in enumerate(sets, 1):
            parts = str(text).replace(' ', '').split(',')
            if len(parts) != 2:
                raise ValueError(f"Set {k} must be 'N_a, N_d', e.g. '1e16, 1e15'; got {text!r}.")
            kw['sets'].append(tuple(to_float(v, f'set {k}') for v in parts))
    if kw.get('npts') == 0:
        kw['npts'] = None                                     # Automatic grid
    return kw


_PN_ROWS = ['W', 'W_box', 'Emax', 'Emax_2kT']

CASES = {
    '6-A': dict(title='pn junction at equilibrium', runner=run_junction, fixed=dict(analysis='point', V=0.0),
                summary='Charge, field and potential compared with the depletion approximation.',
                params=dict(Na=1e16, Nd=1e15, Lp_um=2.0, Ln_um=5.0, T=300.0, srh=True, thr=0.5, npts=1401),
                rows=['Vbi', 'ni', 'xn', 'xp'] + _PN_ROWS + ['LD_p', 'LD_n', 'xp_LD'],
                panels=['rho', 'E', 'psi', 'band', 'density', 'neutrality'],
                schematic=lambda v: schematic_pn(biased=False)),
    '6-B': dict(title='Symmetric vs asymmetric doping', runner=run_doping_sets,
                summary='Three doping sets at equilibrium: where the depletion approximation fails.',
                params=dict(set1='1e16, 1e16', set2='1e16, 1e15', set3='1e17, 1e15', Lp_um=1.0, Ln_um=5.0,
                            srh=True, thr=0.5, T=300.0, npts=3001),
                rows=None, panels=['E_norm', 'rho_norm'], schematic=lambda v: schematic_doping_sets()),
    '6-C': dict(title='Bias sweep (reverse to forward)', runner=run_junction, fixed=dict(analysis='sweep'),
                summary='One sweep from V = −VR,max to +Vmax: J–V, ideality factor, W and |E|max.',
                params=dict(Na=1e16, Nd=1e16, tau_us=0.001, VR_max=5.0, VR_step=0.5, V_max=0.7, V_step=0.05,
                            Lp_um=10.0, Ln_um=10.0, srh=True, from_equilibrium=False, thr=0.5, T=300.0, npts=1601),
                rows=['Vbi', 'Js', 'Ln_Lp', 'W_neutral'],
                cols=['V', 'J', 'J_diode', 'J_model', 'ideality', 'W_sim', 'W_box', 'W_dep', 'Emax_sim', 'Emax_dep',
                      'iters'],
                panels=['JV_lin', 'JV', 'ideality', 'W', 'Emax', 'W2', 'band', 'density', 'rho', 'E', 'current',
                        'rate'],
                schematic=lambda v: schematic_pn(biased=True)),
}

# (2) Visualization & interactive widget

In [ ]:
import contextlib
import html
import io
import time
import traceback

import ipywidgets as widgets
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

###############################################################################
# Plot Settings and Figure Builder
###############################################################################

C_E, C_H, C_T, C_A = '#1f77b4', '#d62728', '#222222', '#7f7f7f'   # Electrons, holes, total, analytic
FIG_WIDTH = 1150
PANEL_HEIGHT = 250         # Height of one panel [px]
ROW_GAP = 105              # Vertical gap between panel rows (axis title + panel title) [px]
NMAX_PLOT = 600            # Maximum points per curve sent to the browser
NMAX_FRAME = 300           # Maximum points per curve in each bias-sweep frame


def _sub(s):
    """Subscripts in Plotly text: 'E_g', 'μ_n', 'N_{a}' -> 'E<sub>g</sub>', 'μ<sub>n</sub>', 'N<sub>a</sub>'."""
    if not isinstance(s, str):
        return s
    return re.sub(r'(?<=[^\W\d_])_(\{[^}]*\}|[A-Za-z0-9]+)', lambda m: f"<sub>{m.group(1).strip('{}')}</sub>", s)


def _trace(x, y, name, color=None, dash='solid', mode='lines', width=2):
    """Trace specification."""
    return dict(x=np.asarray(x, dtype=float), y=np.asarray(y, dtype=float), name=name, color=color,
                dash=dash, mode=mode, width=width)


def _panel(title, xlabel, ylabel, traces, ylog=False, xrange=None, yrange=None, vlines=(), xlog=False, decades=None):
    """Panel specification; decades limits a log axis to that many decades below the maximum."""
    return dict(title=title, xlabel=xlabel, ylabel=ylabel, traces=traces, ylog=ylog,
                xrange=xrange, yrange=yrange, vlines=list(vlines), xlog=xlog, decades=decades)


def _thin(x, y, ylog, nmax=NMAX_PLOT):
    """Reduce points along the arc length of the curve, keeping steep regions dense."""
    if len(x) <= nmax:
        return x, y
    yy = np.log10(np.where(y > 0, y, np.nan)) if ylog else y.astype(float)
    ok = np.isfinite(yy)
    if ok.sum() < 2:
        idx = np.linspace(0, len(x) - 1, nmax).astype(int)
        return x[idx], y[idx]
    yy = np.where(ok, yy, np.nanmin(yy[ok]))
    xs = (x - x.min()) / max(np.ptp(x), 1e-30)
    ys = (yy - yy.min()) / max(np.ptp(yy), 1e-30)
    s = np.concatenate([[0.0], np.cumsum(np.hypot(np.diff(xs), np.diff(ys)))])
    idx = np.unique(np.concatenate([np.searchsorted(s, np.linspace(0, s[-1], nmax)),
                                    np.linspace(0, len(x) - 1, nmax // 4).astype(int), [len(x) - 1]]))
    idx = idx[idx < len(x)]
    return x[idx], y[idx]


def _scatter(t, ylog, legend, nmax=NMAX_PLOT, xrange=None):
    ylog = ylog or t.get('thin_log', False)
    x, y = t['x'], (np.where(t['y'] > 0, t['y'], np.nan) if ylog else t['y'])
    if xrange is not None and len(x) > 2:
        # Keep the zoomed window (plus one point on each side) before thinning.
        inside = np.flatnonzero((x >= xrange[0]) & (x <= xrange[1]))
        if inside.size:
            sl = slice(max(inside[0] - 1, 0), min(inside[-1] + 2, len(x)))
            x, y = x[sl], y[sl]
    x, y = _thin(x, y, ylog, t.get('nmax', nmax))
    extra = {k: t[k] for k in ('visible', 'legendgroup', 'showlegend', 'fill', 'fillcolor', 'opacity') if k in t}
    line = dict(color=t['color'], dash=t['dash'], width=t['width'])
    if 'shape' in t:
        line['shape'] = t['shape']
    return go.Scatter(x=x.astype(np.float32), y=y.astype(np.float32), name=_sub(t['name']), mode=t['mode'], line=line,
                      marker=dict(size=t.get('marker_size', 7), color=t['color']), legend=legend, **extra)


def _log_range(values, decades=30):
    v = np.concatenate([np.asarray(a, dtype=float).ravel() for a in values])
    v = v[np.isfinite(v) & (v > 0)]
    if v.size == 0:
        return None
    hi = np.log10(v.max())
    lo = max(np.log10(v.min()), hi - decades)
    if hi - lo < 1.0:                                   # Nearly constant: two decades around the values
        c = 0.5 * (hi + lo)
        return [c - 1.0, c + 1.0]
    pad = 0.05 * max(hi - lo, 1.0)
    return [lo - pad, hi + pad]


def _lin_range(values, symmetric=False):
    v = np.concatenate([np.asarray(a, dtype=float).ravel() for a in values])
    v = v[np.isfinite(v)]
    if v.size == 0:
        return None
    lo, hi = (-np.max(np.abs(v)), np.max(np.abs(v))) if symmetric else (v.min(), v.max())
    if hi - lo <= 1e-6 * max(abs(lo), abs(hi)):          # Nearly constant: include zero
        lo, hi = min(lo, 0.0), max(hi, 0.0)
        if hi == lo:
            return [-1.0, 1.0]
    pad = 0.05 * (hi - lo)
    return [lo - pad, hi + pad]


def build_figure(panels, title, ncols=2, bottom=60):
    """
    Build one Plotly figure with one legend per panel (placed to the right of the panel).

    output:
        go.Figure   fig
        list        trace_ids   Trace indices of each panel
    """
    nrows = (len(panels) + ncols - 1) // ncols
    plot_height = nrows * PANEL_HEIGHT + (nrows - 1) * ROW_GAP
    fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=[_sub(pn['title']) if pn else ' ' for pn in panels],
                        horizontal_spacing=0.24, vertical_spacing=ROW_GAP / plot_height if nrows > 1 else 0.0)
    trace_ids = []
    for k, pn in enumerate(panels):
        r, c = k // ncols + 1, k % ncols + 1
        if pn is None:                              # Blank position (keeps the following panels aligned)
            fig.update_xaxes(visible=False, row=r, col=c)
            fig.update_yaxes(visible=False, row=r, col=c)
            trace_ids.append([])
            continue
        legend = 'legend' if k == 0 else f'legend{k + 1}'
        ids = []
        for t in pn['traces']:
            fig.add_trace(_scatter(t, pn['ylog'], legend, xrange=pn['xrange']), row=r, col=c)
            ids.append(len(fig.data) - 1)
        trace_ids.append(ids)
        sub = fig.get_subplot(r, c)
        fig.update_layout({legend: dict(x=sub.xaxis.domain[1] + 0.01, y=sub.yaxis.domain[1], xanchor='left',
                                        yanchor='top', font=dict(size=10), bgcolor='rgba(255,255,255,0.85)')})
        yrange = pn['yrange']
        if yrange is None:                          # Limit log axes; avoid zoomed-in axes on constant curves
            vals = [t['y'] for t in pn['traces']]
            if pn['ylog']:
                auto = _log_range(vals, decades=pn.get('decades') or 30)
                if auto is not None and (pn.get('decades') or auto[1] - auto[0] <= 2.0):
                    yrange = auto
            elif vals:
                auto = _lin_range(vals)
                v = np.concatenate([np.asarray(a, dtype=float).ravel() for a in vals])
                v = v[np.isfinite(v)]
                if v.size and np.ptp(v) <= 1e-6 * np.max(np.abs(v)):
                    yrange = auto
        fig.update_xaxes(title_text=_sub(pn['xlabel']), range=pn['xrange'], row=r, col=c)
        fig.update_yaxes(title_text=_sub(pn['ylabel']), range=yrange, row=r, col=c)
        if 'cm⁻³' in pn['ylabel'] or 'A/cm²' in pn['ylabel']:   # 1e16 -> 10^16 instead of SI prefixes (T, p)
            fig.update_yaxes(exponentformat='power', row=r, col=c)
        if pn['ylog']:
            fig.update_yaxes(type='log', exponentformat='power', row=r, col=c)
        if pn.get('xlog'):
            fig.update_xaxes(type='log', exponentformat='power', row=r, col=c)
        for xv in pn['vlines']:
            fig.add_vline(x=xv, line=dict(color='gray', dash='dot', width=1), row=r, col=c)
    fig.update_layout(width=FIG_WIDTH, height=plot_height + 90 + bottom, template='plotly_white', title=_sub(title),
                      margin=dict(l=80, r=200, t=90, b=bottom))
    fig.update_annotations(font_size=13)
    return fig, trace_ids


FIGURE_SINK = None          # Set by the launcher to collect figures instead of displaying them


def show_figure(fig):
    if FIGURE_SINK is not None:
        FIGURE_SINK(fig)
    else:
        display(HTML(fig.to_html(full_html=False, include_plotlyjs='inline', auto_play=False)))


PLOTLY_JS_SENT = {'done': False}     # The launcher embeds plotly.js in its first figure only


def figure_light_html(fig):
    """Figure HTML without plotly.js: draws with the plotly.js already in the page, or loads it from the
    plotly CDN if the page does not have it (for example after reloading the window)."""
    html_fig = fig.to_html(full_html=False, include_plotlyjs=False, auto_play=False)
    cdn = re.search(r'src="([^"]+)"', go.Figure().to_html(full_html=False, include_plotlyjs='cdn')).group(1)
    i0 = html_fig.index('<script>') + len('<script>')
    i1 = html_fig.rindex('</script>')
    wrapped = ("(function () { var draw = function () {" + html_fig[i0:i1] + "};"
               " if (window.Plotly) { draw(); return; }"
               " var s = document.createElement('script'); s.src = '" + cdn + "'; s.onload = draw;"
               " document.head.appendChild(s); })();")
    return html_fig[:i0] + wrapped + html_fig[i1:]


def launcher_figure_html(fig):
    """
    Figure HTML for the launcher: (html shown now, html kept for showing again).
    The first figure after launch() embeds plotly.js (about 4.7 MB); later figures carry only their data.
    """
    light = figure_light_html(fig)
    if not PLOTLY_JS_SENT['done']:
        PLOTLY_JS_SENT['done'] = True
        return fig.to_html(full_html=False, include_plotlyjs='inline', auto_play=False), light
    return light, light


def _overlay_traces(ov, key, x0=0.0):
    """Analytic overlays (computed on the device grid) shifted to the plotting origin x0."""
    return [_trace(np.asarray(x) - x0, y, name, C_A, dash, width=1.5) for x, y, name, dash in ov.get(key, [])]


def _rgba(hex_color, alpha):
    h = hex_color.lstrip('#')
    return f'rgba({int(h[0:2], 16)},{int(h[2:4], 16)},{int(h[4:6], 16)},{alpha})'


###############################################################################
# Device Schematics (inline SVG shown next to the case parameters)
###############################################################################

# Drawings with symbols only: the left contact is the bias terminal V, the right contact is grounded.

SCH_W, SCH_H = 480, 225
SCH_FONT = "font-family='Helvetica, Arial, sans-serif'"
SCH_COLORS = dict(p='#d6e4fb', n='#fbd5d5', p_dark='#3b6fd8', n_dark='#d24b4b', edge='#555555', wire='#333333',
                  contact='#444444', light='#e08a00', text='#222222', note='#666666')     # p: blue, n: red
_X0, _X1, _Y0, _Y1 = 110, 370, 95, 140          # Device block [px]


def _markup(s, size):
    """'N_{a}' and 'e^{-s/λ}' -> SVG tspans with subscripts and superscripts."""
    small = round(0.72 * size)
    s = re.sub(r'_\{([^}]*)\}', lambda m: f"<tspan baseline-shift='sub' font-size='{small}'>{m.group(1)}</tspan>", s)
    return re.sub(r'\^\{([^}]*)\}', lambda m: f"<tspan baseline-shift='super' font-size='{small}'>{m.group(1)}</tspan>", s)


def _text(x, y, s, size=13, anchor='middle', color=None, weight='normal', italic=False, bg=False):
    color = color or SCH_COLORS['text']
    style = " font-style='italic'" if italic else ''
    out = ''
    if bg:                                       # White box behind the label (hides lines underneath)
        w = 0.62 * size * len(re.sub(r'[_^]\{([^}]*)\}', r'\1', s)) + 6
        x_left = x - w / 2 if anchor == 'middle' else (x - 3 if anchor == 'start' else x - w + 3)
        out += f"<rect x='{x_left:.1f}' y='{y - size + 1:.1f}' width='{w:.1f}' height='{size + 4}' fill='white'/>"
    return out + (f"<text x='{x}' y='{y}' text-anchor='{anchor}' font-size='{size}' fill='{color}' "
                  f"font-weight='{weight}'{style} {SCH_FONT}>{_markup(s, size)}</text>")


def _line(x0, y0, x1, y1, color=None, width=1.5, dash=None):
    d = f" stroke-dasharray='{dash}'" if dash else ''
    return (f"<line x1='{x0}' y1='{y0}' x2='{x1}' y2='{y1}' stroke='{color or SCH_COLORS['wire']}' "
            f"stroke-width='{width}'{d}/>")


def _arrowhead(x, y, angle, size=7, color=None):
    """Filled arrowhead with its tip at (x, y), pointing along angle [rad]."""
    c, s = np.cos(angle), np.sin(angle)
    pts = [(x, y), (x - size * c + 0.5 * size * s, y - size * s - 0.5 * size * c),
           (x - size * c - 0.5 * size * s, y - size * s + 0.5 * size * c)]
    return (f"<polygon points='{' '.join(f'{px:.1f},{py:.1f}' for px, py in pts)}' "
            f"fill='{color or SCH_COLORS['wire']}'/>")


def _dimension(x0, x1, y, label):
    """Double-headed length arrow with a label underneath."""
    return (_line(x0 + 1, y, x1 - 1, y, width=1) + _arrowhead(x0, y, np.pi) + _arrowhead(x1, y, 0.0)
            + _line(x0, y - 6, x0, y + 6, width=1) + _line(x1, y - 6, x1, y + 6, width=1)
            + _text((x0 + x1) / 2, y + 18, label, bg=True))


def _wavy_arrow(x0, y0, x1, y1, color=None, waves=3, amp=4.0):
    """Light ray: sine-wave path from (x0, y0) to (x1, y1) with an arrowhead."""
    color = color or SCH_COLORS['light']
    L = np.hypot(x1 - x0, y1 - y0)
    ux, uy = (x1 - x0) / L, (y1 - y0) / L
    t = np.linspace(0, L - 8, 60)
    off = amp * np.sin(2 * np.pi * waves * t / (L - 8))
    xs, ys = x0 + ux * t - uy * off, y0 + uy * t + ux * off
    path = 'M ' + ' L '.join(f'{a:.1f},{b:.1f}' for a, b in zip(xs, ys))
    return (f"<path d='{path}' fill='none' stroke='{color}' stroke-width='1.8'/>"
            + _arrowhead(x1, y1, np.arctan2(uy, ux), size=8, color=color))


def _ground(x, y):
    return (_line(x, y, x + 16, y) + _line(x + 16, y, x + 16, y + 26)
            + _line(x + 4, y + 26, x + 28, y + 26, width=2) + _line(x + 8, y + 31, x + 24, y + 31, width=2)
            + _line(x + 12, y + 36, x + 20, y + 36, width=2) + _text(x + 16, y + 52, 'ground', size=11,
                                                                       color=SCH_COLORS['note']))


def _contacts():
    """Thin ohmic-contact plates at both ends of the device."""
    return (f"<rect x='{_X0 - 6}' y='{_Y0 - 4}' width='6' height='{_Y1 - _Y0 + 8}' fill='{SCH_COLORS['contact']}'/>"
            f"<rect x='{_X1}' y='{_Y0 - 4}' width='6' height='{_Y1 - _Y0 + 8}' fill='{SCH_COLORS['contact']}'/>")


def _circuit():
    """Bias terminal V on the left contact and ground on the right contact."""
    ym = (_Y0 + _Y1) / 2
    return (_line(46, ym, _X0 - 6, ym)
            + f"<circle cx='40' cy='{ym}' r='5.5' fill='white' stroke='{SCH_COLORS['wire']}' stroke-width='1.5'/>"
            + _text(40, ym - 14, 'V', weight='bold') + _ground(_X1 + 6, ym))


def _block(x0, x1, fill, label_lines, y0=_Y0, y1=_Y1):
    out = f"<rect x='{x0}' y='{y0}' width='{x1 - x0}' height='{y1 - y0}' fill='{fill}' stroke='{SCH_COLORS['edge']}'/>"
    yc = (y0 + y1) / 2
    for k, s in enumerate(label_lines):
        out += _text((x0 + x1) / 2, yc + 5 + (k - (len(label_lines) - 1) / 2) * 16, s,
                     weight='bold' if k == 0 else 'normal')
    return out


def _origin(x, label='x = 0', through=False):
    """Dashed mark of x = 0; through=True crosses the device (junction), otherwise it starts below the block."""
    y_top = _Y0 - 8 if through else _Y1
    return _line(x, y_top, x, 196, color='#888888', width=1, dash='4,3') + _text(x, 210, label, size=12, bg=True)


def _svg(body, gradient='', top=0):
    """White card with the drawing, shifted up by 'top' px when nothing is drawn above the device."""
    h = SCH_H - top
    return (f"<svg xmlns='http://www.w3.org/2000/svg' width='{SCH_W}' height='{h}' viewBox='0 0 {SCH_W} {h}'>"
            f"<defs>{gradient}</defs>"
            f"<rect x='0.5' y='0.5' width='{SCH_W - 1}' height='{h - 1}' rx='8' fill='white' stroke='#d0d0d0'/>"
            f"<g transform='translate(0,{-top})'>{body}</g></svg>")


def _bar(dtype, label2='N'):
    """Uniformly doped bar: n-type red, p-type blue."""
    return _block(_X0, _X1, SCH_COLORS[dtype], [f'{dtype}-type', label2])


def _pn(biased):
    xm = (_X0 + _X1) / 2
    body = (_block(_X0, xm, SCH_COLORS['p'], ['p-type', 'N_{a}']) + _block(xm, _X1, SCH_COLORS['n'], ['n-type', 'N_{d}'])
            + _contacts() + (_circuit() if biased else '') + _origin(xm, through=True)
            + _dimension(_X0, xm, 160, 'L_{p}') + _dimension(xm, _X1, 160, 'L_{n}'))
    return _svg(body, top=45)


def _glow(gid, width_frac=0.35):
    """Orange shading that fades from the left end of the bar (excess carriers near the left end)."""
    grad = (f"<linearGradient id='{gid}' x1='0' x2='1' y1='0' y2='0'>"
            f"<stop offset='0' stop-color='{SCH_COLORS['light']}' stop-opacity='0.75'/>"
            f"<stop offset='1' stop-color='{SCH_COLORS['light']}' stop-opacity='0'/></linearGradient>")
    rect = (f"<rect x='{_X0}' y='{_Y0}' width='{width_frac * (_X1 - _X0):.0f}' height='{_Y1 - _Y0}' "
            f"fill='url(#{gid})'/>")
    return grad, rect


def _open_bar(dtype, label2='N'):
    """Bar cut at both ends (the sample continues: no contacts)."""
    fill, edge = SCH_COLORS[dtype], SCH_COLORS['edge']
    zig = lambda x, s: ' '.join(f'{x + s * (5 if k % 2 else 0)},{_Y0 + k * (_Y1 - _Y0) / 6:.1f}' for k in range(7))
    body = (f"<rect x='{_X0}' y='{_Y0}' width='{_X1 - _X0}' height='{_Y1 - _Y0}' fill='{fill}'/>"
            + _line(_X0, _Y0, _X1, _Y0, edge, 1) + _line(_X0, _Y1, _X1, _Y1, edge, 1)
            + f"<polyline points='{zig(_X0, -1)}' fill='none' stroke='{edge}' stroke-width='1.2'/>"
            + f"<polyline points='{zig(_X1, 1)}' fill='none' stroke='{edge}' stroke-width='1.2'/>")
    ym = (_Y0 + _Y1) / 2 + 5
    body += (_text(_X0 - 26, ym, '⋯', size=18, color=SCH_COLORS['note'])
             + _text(_X1 + 26, ym, '⋯', size=18, color=SCH_COLORS['note']))
    yc = (_Y0 + _Y1) / 2
    for k, s in enumerate([f'{dtype}-type', label2]):
        body += _text((_X0 + _X1) / 2, yc + 5 + (k - 0.5) * 16, s, weight='bold' if k == 0 else 'normal')
    return body


_XM = (_X0 + _X1) / 2


###############################################################################
# Lab 6 Panels and Figures
###############################################################################

def _junction_zoom(st, p):
    """x range around the depletion region of the abrupt pn junction [um], with x = 0 at the junction."""
    if p['structure'] != 3:
        return None
    c = depletion_compare(st, p['Na'], p['Nd'], p['Lp_um'], p['thr'], st.V)
    xp, xn = (c['xp_a'], c['xn_a']) if c['valid'] else (c['xp_s'], c['xn_s'])
    W = max(xp + xn, 1e-3)
    return [max(st.xc[0], -2.5 * max(xp, 0.1 * W)), min(st.xc[-1], 2.5 * max(xn, 0.1 * W))]


def band_panel(st, title=None):
    title = title or f'Energy band diagram (E_g = {st.dev.Eg:g} eV)'
    return _panel(title, 'x [µm]', 'Energy [eV]', [
        _trace(st.xc, st.Ec, 'E_c', C_T), _trace(st.xc, st.Ev, 'E_v', C_T),
        _trace(st.xc, st.Ei, 'E_Fi', C_A, 'dot', width=1.5),
        _trace(st.xc, st.Fn, 'E_Fn', C_E, 'dash'), _trace(st.xc, st.Fp, 'E_Fp', C_H, 'dash')])


def density_panel(st, title='Carrier and doping concentrations'):
    d = st.dev
    return _panel(title, 'x [µm]', 'Concentration [cm⁻³]', [
        _trace(st.xc, st.n, 'n', C_E), _trace(st.xc, st.p, 'p', C_H),
        _trace(st.xc, d.Nd, 'N_d', C_E, 'dot', width=1.5), _trace(st.xc, d.Na, 'N_a', C_H, 'dot', width=1.5)], ylog=True)


def charge_panel(st, ov, xr=None, xj=None, title='Net charge density ρ/q'):
    return _panel(title, 'x [µm]', 'ρ/q [cm⁻³]', [_trace(st.xc, st.rho, 'p − n + N_d − N_a', C_T)]
                  + _overlay_traces(ov, 'rho', st.dev.x0_um), xrange=xr, vlines=() if xj is None else (xj,))


def field_panel(st, ov, xr=None, title='Electric field'):
    return _panel(title, 'x [µm]', 'E [V/cm]', [_trace(st.xc, st.E, 'simulation', C_T)] + _overlay_traces(ov, 'E', st.dev.x0_um),
                  xrange=xr)


def potential_panel(st, ov, title='Electrostatic potential'):
    return _panel(title, 'x [µm]', 'ψ [V]', [_trace(st.xc, st.psi, 'simulation', C_T)] + _overlay_traces(ov, 'psi', st.dev.x0_um))


def current_panel(st, title='Current density components'):
    return _panel(title, 'x [µm]', 'J [A/cm²]', [
        _trace(st.xhc, st.Jn_drift, 'J_n drift', C_E), _trace(st.xhc, st.Jn_diff, 'J_n diffusion', C_E, 'dash'),
        _trace(st.xhc, st.Jp_drift, 'J_p drift', C_H), _trace(st.xhc, st.Jp_diff, 'J_p diffusion', C_H, 'dash'),
        _trace(st.xhc, st.Jt, 'J total', C_T, width=3)])


def excess_panel(st, ov, title='Excess carriers |δn|, |δp|'):
    # Hide values below the Gummel tolerance (numerical noise at or near equilibrium).
    ni = st.dev.ni
    mask = lambda v, eq: np.where(np.abs(v) > 10 * SIM_CFG.tol_carrier * (eq + ni), np.abs(v), np.nan)
    return _panel(title, 'x [µm]', '|δ| [cm⁻³]', [_trace(st.xc, mask(st.dn, st.n - st.dn), '|δn|', C_E),
                                             _trace(st.xc, mask(st.dp, st.p - st.dp), '|δp|', C_H)]
             + _overlay_traces(ov, 'excess', st.dev.x0_um), ylog=True, decades=8)


def rate_panel(st, title='Generation and recombination rates'):
    G = np.asarray(st.G)
    ylog = G.max() > 10 * max(G.min(), 1e-300)                 # Absorption profile: log axis
    return _panel(title, 'x [µm]', 'Rate [cm⁻³ s⁻¹]', [_trace(st.xc, st.R, 'R (SRH)', C_T),
                                                      _trace(st.xc, G, 'G', C_A, 'dash')], ylog=ylog, decades=6)


def neutrality_panel(st, xr=None, xj=None, title='Charge relative to local doping ρ/(qN)'):
    d = st.dev
    Nloc = np.maximum(np.maximum(d.Na, d.Nd), d.ni)
    return _panel(title, 'x [µm]', 'ρ/(qN)', [_trace(st.xc, st.rho / Nloc, 'simulation', C_T)], xrange=xr,
                  vlines=() if xj is None else (xj,))


def history_panel(st, title='Gummel convergence history'):
    h = np.array(st.hist)
    return _panel(title, 'Iteration', 'Change', [
        _trace(h[:, 0], h[:, 1], 'max|Δψ| [V]', C_T, mode='lines+markers'),
        _trace(h[:, 0], h[:, 2], 'max|Δn|/(n+n_i)', C_E, mode='lines+markers'),
        _trace(h[:, 0], h[:, 3], 'max|Δp|/(p+n_i)', C_H, mode='lines+markers')], ylog=True)


def _run_title(res):
    p, d = res.params, res.device
    title = f"[{res.case}] " if res.case else ''
    title += f"{STRUCTURES[p['structure']]} | {d.material.name}, T = {d.T:g} K"
    if p['structure'] == 3:
        title += f" | N_a = {p['Na']:.3g}, N_d = {p['Nd']:.3g} cm⁻³"
    return title


SWEEP_SUMMARY = ('JV', 'JV_lin', 'ideality', 'W', 'Emax', 'W2', 'iters')


SET_COLORS = ['#1f77b4', '#2ca02c', '#d62728', '#9467bd', '#ff7f0e']


def _sweep_profile_builders(st, ov, xr, xj, model_name):
    """Profile panels of one bias; every bias gives the same traces (needed for frames)."""
    def model(key):
        t = (_overlay_traces(ov, key, st.dev.x0_um)[:1] or [_trace([np.nan], [np.nan], '', C_A, 'dash', width=1.5)])[0]
        t['name'] = model_name                    # Same legend entry in every frame
        return [t]

    def with_model(pn, key):
        pn['traces'] += model(key)
        return pn
    return {'band': lambda: band_panel(st), 'density': lambda: density_panel(st),
            'rho': lambda: with_model(charge_panel(st, {}, xr, xj), 'rho'),
            'E': lambda: with_model(field_panel(st, {}, xr), 'E'),
            'current': lambda: current_panel(st), 'rate': lambda: rate_panel(st), 'history': lambda: history_panel(st)}


SWEEP_STATIC = {'density': (2, 3), 'rate': (1,)}          # Doping and G do not change with bias


SWEEP_FIXED_RANGE = {'band': 'lin', 'density': 'log', 'rho': 'sym', 'E': 'lin'}   # Others: range per frame


def plot_sweep(res, panels=None):
    p, data = res.params, res.data
    Vs, J, states = data['V'], data['J'], data['states']
    junction = 'W_sim' in data
    reverse = junction and Vs.max() <= 0                  # Reverse-bias sweep: plot against V_R = -V
    xb, xlab = (0.0 - Vs, 'V_R [V]') if reverse else (Vs, 'V [V]')       # 0.0 - V avoids -0.00
    blab = (lambda k: f'V_R = {xb[k]:.2f} V') if reverse else (lambda k: f'V = {Vs[k]:+.3f} V')
    order = np.argsort(-Vs, kind='stable') if reverse else np.arange(len(Vs))
    # |J| on log axes; J = 0 at V = 0 without excitation is not plotted.
    absJ = np.where((Vs == 0) & (p['G'] == 0) & (p['inj'] == 0), np.nan, np.abs(J))
    fw = Vs > 0
    lin = [_trace(xb, J, 'simulation', C_T, mode='lines+markers')]
    if 'J_ohmic' in data:
        lin.append(_trace(xb, data['J_ohmic'], 'J = σV/L', C_A, 'dash'))
    if junction:
        lin += [_trace(xb, data['J_diode'], 'ideal diode', C_E, 'dash'),
                _trace(xb, data['J_model'], 'ideal + J_rg', C_A, 'dot')]
    sim = lambda y: _trace(xb, y, 'simulation', C_T, mode='lines+markers')
    dep = lambda y: _trace(xb, y, 'depletion approx.', C_A, 'dash')
    dep2 = lambda y: _trace(xb, y, 'depl. approx., −2kT/q', C_A, 'dot')
    summary = {
        'JV': lambda: _panel('|J|–V', xlab, '|J| [A/cm²]', [sim(absJ)] + ([
            _trace(xb, np.abs(data['J_diode']), 'ideal diode', C_E, 'dash'),
            _trace(xb, np.abs(data['J_rg']), 'recomb./gen. J_rg', C_H, 'dash'),
            _trace(xb, np.abs(data['J_model']), 'ideal + J_rg', C_A, 'dot')] if junction else []), ylog=True),
        'JV_lin': lambda: _panel('J–V', xlab, 'J [A/cm²]', lin, yrange=_lin_range([J])),   # Range of the simulation
        'ideality': lambda: _panel('Local ideality factor', 'V [V]', 'n = (1/V_t) dV/d(ln J)', [
            _trace(Vs[fw], data['ideality'][fw], 'simulation', C_T, mode='lines+markers')]),
        'W': lambda: _panel('Depletion width', xlab, 'W [µm]', [
            _trace(xb, data['W_sim'], 'sim: |ρ| > θqN', C_T, mode='lines+markers'),
            _trace(xb, data['W_box'], 'sim: equivalent box', C_E, mode='lines+markers'),
            dep(data['W_dep']), dep2(data['W_2kT'])]),
        'Emax': lambda: _panel('Maximum electric field', xlab, '|E|max [V/cm]',
                               [sim(data['Emax_sim']), dep(data['Emax_dep']), dep2(data['Emax_2kT'])]),
        'W2': lambda: _panel(f"W² vs {'V_R' if reverse else 'V'} (equivalent box)", xlab, 'W² [µm²]', [
            _trace(xb, data['W_box'] ** 2, 'sim: equivalent box', C_E, mode='lines+markers'),
            dep(data['W_dep'] ** 2), dep2(data['W_2kT'] ** 2)]),
        'iters': lambda: _panel('Gummel iterations', xlab, 'Iterations', [sim(data['iters'])])}
    if panels is None:
        panels = (['JV_lin', 'JV', 'ideality', 'W', 'Emax', 'W2'] if junction else ['JV_lin', 'JV', 'ideality', 'iters']) \
            + ['band', 'density', 'rho', 'E', 'current', 'rate']
    if not junction:
        panels = [k for k in panels if k not in ('W', 'Emax', 'W2')]       # Depletion width needs a pn junction
    pre = [k for k in panels if k is None or k in SWEEP_SUMMARY]
    pro = [k for k in panels if k is not None and k not in SWEEP_SUMMARY]
    if len(pre) % 2:
        pre.append(None)                                  # Profiles start on a new row
    # Bias marker: on |J|-V if shown, otherwise on the first summary panel.
    mkey = 'JV' if 'JV' in pre else next(k for k in pre if k)
    mpanel = summary[mkey]()
    mlog, my = mpanel['ylog'], mpanel['traces'][0]['y']
    mx = mpanel['traces'][0]['x']
    mlookup = {float(v): (x_, y_) for v, x_, y_ in zip(Vs if mkey != 'ideality' else Vs[fw], mx, my)}
    marker = lambda k: _trace([mlookup.get(float(Vs[k]), (np.nan, np.nan))[0]],
                              [mlookup.get(float(Vs[k]), (np.nan, np.nan))[1]], 'current bias', '#ff7f0e', mode='markers')
    summary_panels = []
    for k in pre:
        pn = summary[k]() if k else None
        if k == mkey:
            pn['traces'].append(marker(order[0]))
        summary_panels.append(pn)

    # Profiles: one frame per bias.
    xrs = [_junction_zoom(st, p) for st in states]
    xr = None if xrs[0] is None else [min(r[0] for r in xrs), max(r[1] for r in xrs)]
    xj = 0.0 if junction else None                         # Junction at x = 0
    model_name = 'depletion approx.' if junction else 'analytic model'
    profiles = []
    for st, ov in zip(states, data['overlays']):
        b = _sweep_profile_builders(st, ov, xr, xj, model_name)
        profiles.append({k: b[k]() for k in pro})
    for key, kind in SWEEP_FIXED_RANGE.items():
        if key in pro:
            values = [t['y'] for pr in profiles for t in pr[key]['traces']]
            profiles[order[0]][key]['yrange'] = (_log_range(values) if kind == 'log'
                                                 else _lin_range(values, symmetric=(kind == 'sym')))
    fig, ids = build_figure(summary_panels + [profiles[order[0]][k] for k in pro],
                            f'{_run_title(res)} | {blab(order[0])}', bottom=170)
    npre = len(pre)

    def frame_layout(k):
        lay = dict(title=dict(text=_sub(f'{_run_title(res)} | {blab(k)}')))
        for j, key in enumerate(pro):
            if key in SWEEP_FIXED_RANGE:
                continue
            pn = profiles[k][key]
            vals = [t['y'] for t in pn['traces']]
            sub = fig.get_subplot((npre + j) // 2 + 1, (npre + j) % 2 + 1)
            lay[sub.yaxis.plotly_name] = dict(range=_log_range(vals) if pn['ylog'] else _lin_range(vals))
            if key == 'history':
                lay[sub.xaxis.plotly_name] = dict(range=[0.5, len(states[k].hist) + 0.5])
        return lay

    dynamic = lambda key, items: [t for m, t in enumerate(items) if m not in SWEEP_STATIC.get(key, ())]
    dyn_ids = [ids[pre.index(mkey)][-1]] + [i for j, key in enumerate(pro) for i in dynamic(key, ids[npre + j])]
    frames = []
    for n, k in enumerate(order):
        traces = [_scatter(marker(k), mlog, None)]
        for key in pro:
            pn = profiles[k][key]
            traces += [_scatter(t, pn['ylog'], None, NMAX_FRAME, pn['xrange']) for t in dynamic(key, pn['traces'])]
        frames.append(go.Frame(data=traces, traces=dyn_ids, name=str(n), layout=frame_layout(k)))
    fig.frames = frames
    fig.update_layout(frame_layout(order[0]))
    steps = [{"args": [[str(n)], {"mode": "immediate", "frame": {"duration": 0, "redraw": True},
                                  "transition": {"duration": 0}}],
              "label": f"{xb[k]:.2f}" if reverse else f"{Vs[k]:+.2f}", "method": "animate"}
             for n, k in enumerate(order)]
    fig.update_layout(
        updatemenus=[{"type": "buttons", "direction": "left", "showactive": True,
                      "x": 0.0, "y": 0.0, "xanchor": "left", "yanchor": "top", "pad": {"t": 85},
                      "buttons": [
                          {"label": "Play", "method": "animate",
                           "args": [None, {"fromcurrent": True, "frame": {"duration": 400, "redraw": True},
                                           "transition": {"duration": 0}}]},
                          {"label": "Pause", "method": "animate",
                           "args": [[None], {"mode": "immediate", "frame": {"duration": 0, "redraw": False},
                                             "transition": {"duration": 0}}]}]}],
        sliders=[{"active": 0, "x": 0.12, "y": 0.0, "yanchor": "top", "len": 0.78, "pad": {"t": 70},
                  "currentvalue": {"prefix": "Reverse bias V_R [V]: " if reverse else "Bias V [V]: "},
                  "steps": steps}])
    show_figure(fig)


def plot_doping_sets(res, panels=None):
    """Doping-set comparison: normalized field and charge against x (junction at x = 0)."""
    xr = [-0.5, 1.5]
    E_tr, rho_tr = [], []
    for k, ds in enumerate(res.data['sets']):
        st, col = ds['state'], SET_COLORS[k % len(SET_COLORS)]
        d = st.dev
        Nloc = np.maximum(np.maximum(d.Na, d.Nd), d.ni)
        E_tr += [_trace(st.xc, st.E / ds['Emax_dep'], f"{ds['label']}", col),
                 _trace(st.xc, ds['E_dep'] / ds['Emax_dep'], f"{ds['label']} depl.", col, 'dash', width=1.5)]
        rho_tr += [_trace(st.xc, st.rho / Nloc, ds['label'], col)]
    builders = {
        'E_norm': lambda: _panel('Field normalized by |E|max of the depletion approx.', 'x [µm]',
                                 'E / |E|max,depl.', E_tr, xrange=xr),
        'rho_norm': lambda: _panel('Charge relative to local doping (depletion approx.: ±1 boxes)', 'x [µm]',
                                   'ρ/(qN)', rho_tr, xrange=xr, vlines=(0.0,))}
    keys = ['E_norm', 'rho_norm'] if panels is None else panels
    d = res.device
    title = (f"[{res.case}] " if res.case else '') + \
        f"Abrupt pn junction | {d.material.name}, T = {d.T:g} K | doping sets N_a / N_d [cm⁻³]"
    fig, _ = build_figure([builders[k]() if k else None for k in keys], title)
    show_figure(fig)


def plot_operating_point(res, panels=None):
    st, ov, p = res.data['state'], res.data['overlays'], res.params
    xr = _junction_zoom(st, p)
    xj = 0.0                                              # Junction at x = 0
    builders = {
        'band': lambda: band_panel(st), 'density': lambda: density_panel(st),
        'rho': lambda: charge_panel(st, ov, xr, xj), 'E': lambda: field_panel(st, ov, xr),
        'psi': lambda: potential_panel(st, ov), 'current': lambda: current_panel(st),
        'excess': lambda: excess_panel(st, ov), 'rate': lambda: rate_panel(st),
        'neutrality': lambda: neutrality_panel(st, xr, xj), 'history': lambda: history_panel(st)}
    keys = panels or ['rho', 'E', 'psi', 'band', 'density', 'neutrality']
    fig, _ = build_figure([builders[k]() if k else None for k in keys], f'{_run_title(res)} | V = {st.V:+.3f} V')
    show_figure(fig)


def plot_result(res, panels=None):
    """Plot a result; panels = list of panel keys (default: the case's panels)."""
    panels = panels if panels is not None else res.panels
    print('\n(3) Visualizing ...')
    analysis = res.params['analysis']
    if analysis == 'point':
        print('\tBlue = electrons, Red = holes, Black = total / simulation, Gray = analytic model')
        plot_operating_point(res, panels)
    elif analysis == 'doping_sets':
        print('\tSolid = simulation, dashed = depletion approximation; one color per doping set')
        plot_doping_sets(res, panels)
    else:
        print('\tTop: bias-sweep summary. Bottom: internal profiles; use the slider to change the bias.')
        plot_sweep(res, panels)
    print('Done!!')


###############################################################################
# Lab 6 Device Schematics
###############################################################################

def schematic_pn(biased=False):
    """Abrupt pn junction (p on the left, x = 0 at the junction); biased: V on the p-side contact."""
    return _pn(biased)


def schematic_doping_sets():
    """Three abrupt pn junctions with different doping sets."""
    body = ''
    for k, y in enumerate((40, 95, 150)):
        body += (_block(150, 260, SCH_COLORS['p'], [], y0=y, y1=y + 32)
                 + _block(260, 370, SCH_COLORS['n'], [], y0=y, y1=y + 32)
                 + _text(205, y + 21, 'N_{a}', size=12) + _text(315, y + 21, 'N_{d}', size=12)
                 + _text(140, y + 21, f'set {k + 1}', size=12, anchor='end', weight='bold'))
    body += _line(260, 30, 260, 196, color='#888888', width=1, dash='4,3') + _text(260, 210, 'x = 0', size=12, bg=True)
    return _svg(body)

In [ ]:
###############################################################################
# Interactive Launcher (case selector, parameters, device schematic, result area)
###############################################################################

widget_width = '560px'
widget_description_width = '260px'
widget_style = {'description_width': widget_description_width}


def make_widget_layout(display=''):
    return widgets.Layout(width=widget_width, display=display)


def make_dropdown(options, value, description):
    return widgets.Dropdown(options=options, value=value, description=description, style=widget_style,
                            layout=make_widget_layout())


def make_float_slider(value, min, max, step, description, readout_format='.2f'):
    return widgets.FloatSlider(value=value, min=min, max=max, step=step, description=description,
                               continuous_update=False, orientation='horizontal', readout=True,
                               readout_format=readout_format, style=widget_style, layout=make_widget_layout())


def make_text(value, description):
    """Text input for values in scientific notation (e.g. 1e16)."""
    return widgets.Text(value=value, description=description, continuous_update=False, style=widget_style,
                        layout=make_widget_layout())


def make_int_text(value, min, max, description):
    return widgets.BoundedIntText(value=value, min=min, max=max, description=description, style=widget_style,
                                  layout=make_widget_layout())


def set_widget_visible(widget, visible):
    widget.layout.display = '' if visible else 'none'


# Case parameters in display order: doping and geometry, bias and excitation, material and transport, numerics.
# kind: 'sci' (text, scientific notation), 'text', 'float' (min, max, step, format), 'int' (min, max), 'choice'
PARAM_SPECS = {
    'doping_type':      ('Doping type', 'choice', [('n-type', 'n'), ('p-type', 'p')]),
    'N':                ('Doping N [cm⁻³]', 'sci', None),
    'profile':          ('Doping profile', 'choice', [('Exponential  N(0) e^(−s/λ)', 'exponential'),
                                                      ('Linear  N(0) → N(L)', 'linear')]),
    'N0':               ('N(0) at the left contact [cm⁻³]', 'sci', None),
    'NL':               ('N(L) at the right contact [cm⁻³]', 'sci', None),
    'decay_um':         ('Decay length λ [µm]', 'float', (0.05, 50.0, 0.05, '.2f')),
    'N_min':            ('Lowest doping Na [cm⁻³]', 'sci', None),
    'N_max':            ('Highest doping Na [cm⁻³]', 'sci', None),
    'length_um':        ('Length L [µm]', 'float', (0.1, 500.0, 0.1, '.1f')),
    'ends':             ('Ends of the bar', 'choice', [('No contacts (infinite sample)', 'free'),
                                                       ('Ohmic contacts', 'ohmic')]),
    'set1':             ('Set 1: Na, Nd [cm⁻³]', 'text', None),
    'set2':             ('Set 2: Na, Nd [cm⁻³]', 'text', None),
    'set3':             ('Set 3: Na, Nd [cm⁻³]', 'text', None),
    'Na':               ('Na, p side [cm⁻³]', 'sci', None),
    'Nd':               ('Nd, n side [cm⁻³]', 'sci', None),
    'Lp_um':            ('p-side length Lp [µm]', 'float', (0.1, 200.0, 0.1, '.1f')),
    'Ln_um':            ('n-side length Ln [µm]', 'float', (0.1, 200.0, 0.1, '.1f')),
    'E_field':          ('Electric field E (along +x) [V/cm]', 'float', (0.0, 3000.0, 100.0, '.0f')),
    'V':                ('Bias V (left contact) [V]', 'float', (-20.0, 1.5, 0.01, '.2f')),
    'VR_max':           ('Maximum reverse bias VR [V]', 'float', (0.0, 50.0, 0.1, '.1f')),
    'VR_step':          ('VR step [V]', 'float', (0.05, 5.0, 0.05, '.2f')),
    'V_max':            ('Maximum forward bias [V]', 'float', (0.05, 1.5, 0.01, '.2f')),
    'V_step':           ('Forward bias step [V]', 'float', (0.01, 0.5, 0.01, '.2f')),
    'from_equilibrium': ('Initial guess at each bias', 'choice',
                         [('Previous bias (continuation)', False), ('Equilibrium', True)]),
    'G':                ('Generation rate G (G₀ at the surface) [cm⁻³s⁻¹]', 'sci', None),
    'gen_depth_um':     ('Absorption depth d [µm]', 'float', (0.1, 10.0, 0.1, '.1f')),
    'inj':              ('Injection δ(0) at the left contact [cm⁻³]', 'sci', None),
    'material':         ('Material (Eg; μn/μp)', 'choice',
                         [(f'{m}: Eg {MATERIALS[m].Eg:g} eV; μn/μp {MATERIALS[m].mun:g}/{MATERIALS[m].mup:g} cm²/V·s', m)
                          for m in ('Si', 'GaAs', 'Ge')]),
    'T':                ('Temperature [K]', 'float', (200.0, 500.0, 5.0, '.0f')),
    'ni300':            ('ni at 300 K [cm⁻³]', 'sci', None),
    'tau_us':           ('τn = τp [µs]', 'sci', None),
    'mun':              ('μn [cm²/V·s]', 'float', (1.0, 10000.0, 1.0, '.0f')),
    'mup':              ('μp [cm²/V·s]', 'float', (1.0, 10000.0, 1.0, '.0f')),
    't_max_ps':         ('Simulation time [ps]', 'float', (10.0, 40.0, 5.0, '.0f')),
    'frame_ps':         ('Frame interval [ps]', 'float', (0.2, 1.0, 0.1, '.1f')),
    'srh':              ('Recombination (SRH)', 'choice', [('On', True), ('Off', False)]),
    'thr':              ('Depletion edge |ρ|/(qN)', 'float', (0.05, 0.95, 0.05, '.2f')),
    'npts':             ('Grid points (0 = auto)', 'int', (0, 100001)),
}
# Parameters shown only for one value of another parameter: key -> (controlling key, value)
PARAM_WHEN = {'decay_um': ('profile', 'exponential'), 'NL': ('profile', 'linear')}


def make_param_widget(key):
    label, kind, spec = PARAM_SPECS[key]
    if kind == 'choice':
        return make_dropdown(spec, spec[0][1], label)
    if kind in ('sci', 'text'):
        return make_text('', label)
    if kind == 'int':
        return make_int_text(spec[0], spec[0], spec[1], label)
    lo, hi, step, fmt = spec
    return make_float_slider(lo, lo, hi, step, label, readout_format=fmt)


def to_widget_value(key, value):
    if PARAM_SPECS[key][1] == 'sci':
        return f'{value:g}'.replace('e+', 'e')
    return value


def from_widget_value(key, value):
    if PARAM_SPECS[key][1] == 'sci':
        return to_float(value, PARAM_SPECS[key][0])
    return value


UI = {}                                       # Widgets of the launcher: UI['case'], ['controls'], ['button'], ...
LAST = {}                                     # Last result: LAST['res'] (also LAST[case_id] for each case run)
RESULT = {'handle': None}                     # Result area below the launcher


def _log_html(text):
    return (f"<pre style='font-size:12px; line-height:1.35; margin:4px 0; overflow-x:auto'>"
            f"{html.escape(text)}</pre>") if text else ''


def launch(title=''):
    """
    Display the launcher of a lab: Case selector, case parameters, device schematic and Run Interact button,
    with the result area (log and figure) below. Selecting a case loads its presets; the calculation starts
    only with Run Interact. The result area is an ordinary output updated by display id: ipywidgets Output
    widgets are not used, because some front ends (VS Code) hold all kernel messages until an Output-widget
    update is acknowledged, which can stall the notebook.

    input:
        str         title       Heading of the launcher (cases: CASES of this notebook, run with run_case)
    """
    cases, run = CASES, run_case
    ids = list(cases)
    keys = [k for k in PARAM_SPECS if any(k in cases[c]['params'] for c in ids)]
    case_widget = widgets.Dropdown(options=[(f"{c}  {cases[c]['title']}", c) for c in ids], value=ids[0],
                                   description='Case', style={'description_width': '80px'},
                                   layout=widgets.Layout(width='680px'))
    case_info = widgets.HTML()
    controls = {k: make_param_widget(k) for k in keys}
    button = widgets.Button(description='Run Interact', button_style='primary', icon='play',
                            layout=widgets.Layout(width='180px', margin='12px 0 4px 0'))
    status = widgets.HTML(layout=widgets.Layout(margin='12px 0 4px 12px'))
    schematic = widgets.HTML(layout=widgets.Layout(margin='6px 0 0 24px'))      # Device drawing (SVG)
    busy = {'run': False, 't_end': 0.0}

    def apply_case(change=None):
        cid = case_widget.value
        case = cases[cid]
        case_info.value = (f"<div style='margin:2px 0 10px 86px;color:#444'><b>{cid}</b> — "
                           f"{html.escape(case['summary'])}</div>")
        for k, w in controls.items():
            set_widget_visible(w, k in case['params'])
            if k in case['params']:
                if PARAM_SPECS[k][1] == 'float':           # Slider range: case override or default
                    lo, hi, step = case.get('ranges', {}).get(k, PARAM_SPECS[k][2][:3])
                    if lo > w.max:
                        w.max, w.min = hi, lo
                    else:
                        w.min, w.max = lo, hi
                    w.step = step
                w.value = to_widget_value(k, case['params'][k])
        update_visibility()

    def update_visibility(change=None):
        params = cases[case_widget.value]['params']
        for k, (key, value) in PARAM_WHEN.items():
            if k in controls and k in params:
                set_widget_visible(controls[k], key not in controls or controls[key].value == value)
        update_schematic()

    def update_schematic(change=None):
        case = cases[case_widget.value]
        values = {k: controls[k].value for k in case['params'] if PARAM_SPECS[k][1] == 'choice'}
        schematic.value = case['schematic'](values) if case.get('schematic') else ''

    def on_run(_=None):
        # The printed log and the figure are collected into one HTML block that replaces the result area.
        global FIGURE_SINK
        if busy['run'] or time.time() - busy['t_end'] < 0.5:     # Running, or a click queued during the run
            return
        busy['run'], button.disabled = True, True
        status.value = '<i>Running ...</i>'
        shown, log = [], io.StringIO()

        def flush_log():
            shown.append(_log_html(log.getvalue()))
            log.seek(0)
            log.truncate(0)

        def add_figure(fig):
            flush_log()
            shown.append(launcher_figure_html(fig)[0])

        t0 = time.time()
        FIGURE_SINK = add_figure
        try:
            with contextlib.redirect_stdout(log), contextlib.redirect_stderr(log):
                try:
                    cid = case_widget.value
                    values = {k: from_widget_value(k, controls[k].value) for k in cases[cid]['params']}
                    res = run(cid, **values)
                    LAST['res'] = LAST[cid] = res
                    plot_result(res)
                    print(f"(Result stored in LAST['{cid}'] and LAST['res'])")
                except ValueError as err:
                    print(f'Input error: {err}')
                except Exception:
                    traceback.print_exc()
            flush_log()
            if RESULT['handle'] is not None:
                RESULT['handle'].update(HTML(''.join(shown)))
        finally:
            FIGURE_SINK = None
            busy['run'], button.disabled = False, False
            busy['t_end'] = time.time()
            status.value = f'Finished in {time.time() - t0:.1f} s'

    apply_case()
    case_widget.observe(apply_case, names='value')
    for k in ('doping_type', 'ends'):
        if k in controls:
            controls[k].observe(update_schematic, names='value')
    if 'profile' in controls:
        controls['profile'].observe(update_visibility, names='value')
    button.on_click(on_run)
    heading = widgets.HTML(f"<h3 style='margin:4px 0 8px 0'>{html.escape(title)}</h3>") if title else None
    params_box = widgets.VBox([*controls.values(), widgets.HBox([button, status])])
    body = widgets.HBox([params_box, schematic],                  # Drawing to the right of the parameters
                        layout=widgets.Layout(flex_flow='row wrap', align_items='flex-start'))
    panel = widgets.VBox([w for w in (heading, case_widget, case_info, body) if w is not None])
    UI.update(case=case_widget, controls=controls, button=button, panel=panel, run=on_run, schematic=schematic)
    PLOTLY_JS_SENT['done'] = False                            # The next figure embeds plotly.js
    display(panel)
    RESULT['handle'] = display(HTML("<p style='color:#777'><i>Select a case and click Run Interact.</i></p>"),
                               display_id=True)                # Result area, updated by Run Interact

---
# (3) Lab 6 — pn junction: Poisson equation and diode characteristics

The Poisson equation links the space charge of the junction to its field and potential. The depletion approximation replaces the charge by boxes of $-qN_a$ and $+qN_d$ with sharp edges [1, Ch. 7]:

$$V_{bi}=\frac{kT}{e}\ln\frac{N_aN_d}{n_i^2}\ (7.10),\qquad W=\left[\frac{2\varepsilon_s(V_{bi}+V_R)}{e}\,\frac{N_a+N_d}{N_aN_d}\right]^{1/2}\ (7.34),\qquad \lvert\mathcal{E}_{max}\rvert=\left[\frac{2e(V_{bi}+V_R)}{\varepsilon_s}\,\frac{N_aN_d}{N_a+N_d}\right]^{1/2}\ (7.36)$$

- The depletion layer extends mainly into the lightly doped side, $x_n/x_p=N_a/N_d$. In a strongly asymmetric junction the depletion width on the heavily doped side becomes comparable to its Debye length, and the box shape fails; case 6-B needs a fine grid (about 3000 points or more) to resolve it.
- Under reverse bias, $W^2=\frac{2\varepsilon_s}{e}\frac{N_a+N_d}{N_aN_d}(V_{bi}+V_R)$ is linear in $V_R$: the slope gives $N_aN_d/(N_a+N_d)$. With the equivalent box width of the simulated charge, the line meets $W^2=0$ at $V_R\approx-(V_{bi}-2kT/e)$.
- Under forward bias the barrier is lowered and the current follows the ideal-diode law with a recombination contribution at low bias [1, Ch. 8]. With **Recombination (SRH) = Off**, the recombination current vanishes and the diode current is set by the neutral-region widths (short-diode limit).

### Why ρ(x) is not a box

The simulated charge density is the correct solution of the Poisson equation with mobile carriers; the box is the idealization. Near each depletion edge the majority-carrier concentration changes over a few Debye lengths, $L_D=(\varepsilon_skT/e^2N)^{1/2}$ (0.04 µm at $10^{16}$ cm⁻³, 0.13 µm at $10^{15}$ cm⁻³), so the edges of $\rho(x)$ are rounded. On the lightly doped side of an asymmetric junction, holes from the heavily doped side also spill over the junction and make $\rho/q$ exceed $N_d$ next to $x=0$. Away from the edges the simulated charge is flat at $-qN_a$ and $+qN_d$, and the depleted charge corresponds to the depletion approximation with $V_{bi}-V-2kT/e$ (rows `W_box` and `Emax_2kT`; see below).

| Case | Device and condition | Compared with |
| --- | --- | --- |
| **6-A** pn junction at equilibrium | $N_a$, $N_d$, temperature $T$, $V=0$ | $V_{bi}$ (7.10), $x_n$ (7.28), $x_p$ (7.29), $W$ (7.31), $\lvert\mathcal{E}\rvert_{max}$ (7.36), the $2kT/e$-corrected model and the equivalent box width; Debye lengths and $x_p/L_D$, $x_n/L_D$ |
| **6-B** Symmetric vs asymmetric doping | Three doping sets at equilibrium | $W$ and $\lvert\mathcal{E}\rvert_{max}$ of each set; field normalized by the depletion-approximation maximum, $\rho/(qN)$ |
| **6-C** Bias sweep (reverse to forward) | $V=-V_{R,max}$ to $+V_{max}$ in one sweep | $J$–$V$ on linear and log axes with the ideal diode (8.26), (8.27), recombination/generation current (8.55) and their sum (8.56); local ideality factor; $W\propto\sqrt{V_{bi}-V}$ (7.34), $\lvert\mathcal{E}\rvert_{max}$ (7.36), $W^2$ vs $V$ |

### Why $W$ and $\lvert\mathcal{E}\rvert_{max}$ differ from the depletion approximation

The depletion approximation assumes complete depletion with sharp edges. In the simulation the majority-carrier concentration falls off over a few Debye lengths $L_D$ at each edge, so the charge has smooth tails instead of box edges (see the $\rho/q$ panel). Three effects appear in the comparison table:

1. **Majority-carrier tails (physics).** The tails screen part of the junction voltage; the depletion layer behaves as if the voltage were $V_{bi}-V-2kT/e$ instead of $V_{bi}-V$ [2, Sec. 2.2]. For $N_a=N_d=10^{16}$ cm⁻³, $2kT/e=0.052$ V is 7.5 % of $V_{bi}=0.695$ V, which lowers $W$ and $\lvert\mathcal{E}\rvert_{max}$ by about 4 %. The equivalent box width of the simulated charge, $\int\lvert\rho\rvert dx/qN=0.408$ µm (row `W_box`), equals the corrected depletion width 0.408 µm, compared with 0.424 µm from (7.31).
2. **Edge definition.** The simulated $W$ is measured where $\lvert\rho\rvert>\theta qN$ ($\theta=0.5$ by default). Because the edges are smooth, a different $\theta$ gives a different $W$; `W_box` does not depend on $\theta$.
3. **Grid resolution.** $\lvert\mathcal{E}\rvert_{max}$ is a sharp peak at the junction. For $N_a=N_d=10^{16}$ cm⁻³ on a 4 µm device it is 1.2 % below the corrected model with 801 grid points and 0.2 % below with 4001 points (row `Emax_2kT`).

### Temperature

The **Temperature** control changes $kT/e$ and $n_i(T)$ (see **Model assumptions**). Because $n_i$ grows quickly with $T$, $V_{bi}=(kT/e)\ln(N_aN_d/n_i^2)$ falls from 0.86 V at 200 K to 0.34 V at 500 K for Si with $N_a=N_d=10^{16}$ cm⁻³, and $W$ shrinks with it. The saturation current $J_s\propto n_i^2$ and the generation current $J_{gen}\propto n_i$ rise steeply, so the reverse current at 400 K is orders of magnitude larger than at 300 K.

### Gummel iteration: how the modules connect

The unknowns $\psi$, $n$ and $p$ depend on each other: the Poisson equation needs $n$ and $p$, and the current and continuity equations need $\psi$. The solver updates them in turn [3]:

$$\text{①}\ \ n^{(k)},p^{(k)}\ \text{fixed}\ \Rightarrow\ \frac{d^2\psi}{dx^2}=-\frac{q}{\varepsilon_s}\left(p-n+N_d-N_a\right)\ \Rightarrow\ \psi^{(k+1)}\qquad(\texttt{poisson},\ \text{Ch. 7})$$

$$\text{②}\ \ \psi^{(k+1)}\ \text{fixed}\ \Rightarrow\ \frac{1}{q}\frac{dJ_p}{dx}=G-R\ \Rightarrow\ p^{(k+1)},\qquad \text{③}\ \ \frac{1}{q}\frac{dJ_n}{dx}=R-G\ \Rightarrow\ n^{(k+1)}\qquad(\texttt{continuity}+\texttt{drift\_diffusion},\ \text{Chs. 5–6})$$

④ The loop returns to ① until $\max\lvert\Delta\psi\rvert$, $\max\lvert\Delta n\rvert/n$ and $\max\lvert\Delta p\rvert/p$ are small. Steps ② and ③ use the drift–diffusion currents of Lab 4 inside the continuity equations of Lab 5, step ① is the Poisson equation of this lab, and all densities obey the Boltzmann statistics of the `statistics` module. The loop is written out in this notebook (`GummelSolver.solve`). The number of iterations at each bias is printed in the sweep table: as the current grows, the three equations couple more strongly and more iterations are needed; starting each bias from the previous solution (continuation) usually needs fewer iterations than starting from equilibrium.

---
# (4) Simulation

Select a case, edit the parameters and click **Run Interact**.

In [ ]:
launch('Lab 6 — pn junction: Poisson equation and diode')